# synthkit — fully self-contained notebook

The ENTIRE library inlined below, generated from the
module sources by `scripts/build_notebook.py` — nothing
to install, no repo required: run top to bottom in any
JupyterLab (SageMaker Studio included).

Pipeline: plain-English spec -> deterministic planning
(blueprints ARE the ground truth) -> verified rendering
(code verifier around every LLM call, deterministic
fallback) -> exact sliced evaluation -> measured
experiments.

PHI posture: there is NO ingestion path anywhere below.
Distributions are specified, never fitted; nothing real
ever enters the generator.


## Library: The DataSpec — the reviewable contract

*Source of truth: `synthkit/spec.py` — this cell is generated, not hand-edited.*


In [ ]:
"""SYNTH_V1 S1: the DataSpec — the reviewable contract.

Plain English goes into the compiler; a DataSpec comes out; a human
reads and edits it BEFORE anything generates. Every downstream stage
(planner, renderers, evaluator) consumes only this object, so the
spec is the single place where "what should this corpus contain"
lives. JSON round-trip is exact: load(dump(spec)) == spec.

Design rules:
  - No ingestion path. Distributions are SPECIFIED, never fitted
    from data. Nothing real ever enters (the PHI posture is
    structural, not procedural).
  - Validation is loud and total: a spec either validates completely
    or raises SpecError listing every problem found — no partial
    acceptance, because the spec doubles as the evaluation ground
    truth's schema.

Python 3.8 compatible. Stdlib only.
"""
from __future__ import annotations

import json
from dataclasses import asdict, dataclass, field
from typing import Any, Dict, List, Optional

SPEC_VERSION = 1

FIELD_TYPES = ("int", "float", "str", "date", "categorical", "id",
               "person_name")
DISTRIBUTIONS = ("uniform", "normal", "lognormal", "categorical",
                 "date_range", "sequence")
DIFFICULTIES = ("easy", "medium", "hard")


class SpecError(ValueError):
    """Raised with EVERY validation problem, newline-joined."""


# ===================================================================
# Structured side
# ===================================================================

@dataclass
class Distribution:
    """How a structured field's values are drawn.

    kind        one of DISTRIBUTIONS
    params      kind-specific:
      uniform      {"low": x, "high": y}            (int/float)
      normal       {"mean": m, "stdev": s, "min"?: , "max"?: }
      lognormal    {"mu": m, "sigma": s, "min"?: , "max"?: }
      categorical  {"choices": ["a", ...], "weights"?: [0.5, ...]}
      date_range   {"start": "YYYY-MM-DD", "end": "YYYY-MM-DD"}
      sequence     {"prefix"?: "ENC-", "start"?: 1000}   (ids)
    """
    kind: str
    params: Dict[str, Any] = field(default_factory=dict)


@dataclass
class StructuredField:
    name: str
    ftype: str                       # one of FIELD_TYPES
    distribution: Distribution
    nullable_rate: float = 0.0       # fraction of docs where null


@dataclass
class CrossFieldRule:
    """V1 supports ordered pairs: `later` must exceed `earlier` by a
    delta drawn from [min_delta, max_delta] (days for dates, units
    for numerics). Enforced constructively at planning time — the
    planner samples `earlier` and a delta, never rejection-samples."""
    earlier: str
    later: str
    min_delta: float = 0.0
    max_delta: float = 30.0


# ===================================================================
# Unstructured side
# ===================================================================

@dataclass
class TargetElement:
    """A plantable fact the outside model SHOULD extract.

    element_id   stable key, becomes the ground-truth label
    description  what the fact is ("current medication with dose")
    phrasings    2+ surface realizations; the planner draws one per
                 document ("{value} 20mg daily", "pt continues
                 {value}"). '{value}' slots a sampled value when
                 value_source names a structured field or a
                 categorical pool.
    density      fraction of documents that contain this element
    difficulty   easy/medium/hard — a rendering hint (hard = buried,
                 abbreviated, or split across sentences) and an
                 evaluation slice
    value_source optional: structured field name or inline
                 {"choices": [...]} pool supplying '{value}'
    """
    element_id: str
    description: str
    phrasings: List[str]
    density: float = 1.0
    difficulty: str = "medium"
    value_source: Optional[Any] = None


@dataclass
class Distractor:
    """A plausible near-miss the model should NOT extract — the part
    of the spec that makes a test adversarial rather than a demo.
    Same shape as a target minus difficulty semantics."""
    distractor_id: str
    description: str
    phrasings: List[str]
    density: float = 0.5
    value_source: Optional[Any] = None


@dataclass
class StyleAxes:
    """Controlled variation axes; the planner draws one value per
    axis per document, and the evaluator slices results by them."""
    personas: List[str] = field(default_factory=lambda: ["neutral"])
    verbosity: List[str] = field(
        default_factory=lambda: ["terse", "moderate", "verbose"])
    abbreviation: List[str] = field(
        default_factory=lambda: ["none", "moderate", "heavy"])


@dataclass
class UnstructuredField:
    name: str
    note_type: str                   # "nursing progress note", ...
    target_elements: List[TargetElement] = field(default_factory=list)
    distractors: List[Distractor] = field(default_factory=list)
    style: StyleAxes = field(default_factory=StyleAxes)
    length_words: List[int] = field(
        default_factory=lambda: [80, 220])   # [min, max]


# ===================================================================
# The spec
# ===================================================================

@dataclass
class CorpusConfig:
    size: int = 50
    master_seed: int = 20260715


@dataclass
class DataSpec:
    title: str
    structured_fields: List[StructuredField] = field(default_factory=list)
    unstructured_fields: List[UnstructuredField] = field(default_factory=list)
    cross_field_rules: List[CrossFieldRule] = field(default_factory=list)
    corpus: CorpusConfig = field(default_factory=CorpusConfig)
    version: int = SPEC_VERSION

    # ---------------- serialization ----------------

    def to_json(self) -> str:
        return json.dumps(asdict(self), indent=2, ensure_ascii=False)

    @staticmethod
    def from_json(text: str) -> "DataSpec":
        raw = json.loads(text)
        return DataSpec(
            title=raw.get("title", ""),
            structured_fields=[
                StructuredField(
                    name=f["name"], ftype=f["ftype"],
                    distribution=Distribution(**f["distribution"]),
                    nullable_rate=f.get("nullable_rate", 0.0),
                )
                for f in raw.get("structured_fields", [])
            ],
            unstructured_fields=[
                UnstructuredField(
                    name=u["name"], note_type=u.get("note_type", "note"),
                    target_elements=[
                        TargetElement(**t)
                        for t in u.get("target_elements", [])
                    ],
                    distractors=[
                        Distractor(**d) for d in u.get("distractors", [])
                    ],
                    style=StyleAxes(**u.get("style", {})),
                    length_words=u.get("length_words", [80, 220]),
                )
                for u in raw.get("unstructured_fields", [])
            ],
            cross_field_rules=[
                CrossFieldRule(**r)
                for r in raw.get("cross_field_rules", [])
            ],
            corpus=CorpusConfig(**raw.get("corpus", {})),
            version=raw.get("version", SPEC_VERSION),
        )

    # ---------------- validation ----------------

    def validate(self) -> None:
        """Raises SpecError with EVERY problem, or returns quietly."""
        problems: List[str] = []
        names = set()

        for f in self.structured_fields:
            where = "structured field `{}`".format(f.name or "?")
            if not f.name:
                problems.append("structured field with empty name")
            elif f.name in names:
                problems.append("duplicate field name `{}`".format(f.name))
            names.add(f.name)
            if f.ftype not in FIELD_TYPES:
                problems.append("{}: unknown type `{}` (know: {})".format(
                    where, f.ftype, ", ".join(FIELD_TYPES)))
            d = f.distribution
            if d.kind not in DISTRIBUTIONS:
                problems.append("{}: unknown distribution `{}`".format(
                    where, d.kind))
            elif d.kind == "categorical":
                choices = d.params.get("choices") or []
                if not choices:
                    problems.append("{}: categorical needs choices".format(
                        where))
                weights = d.params.get("weights")
                if weights is not None and len(weights) != len(choices):
                    problems.append(
                        "{}: {} weights for {} choices".format(
                            where, len(weights), len(choices)))
            elif d.kind == "uniform":
                if not {"low", "high"} <= set(d.params):
                    problems.append("{}: uniform needs low/high".format(
                        where))
            elif d.kind == "normal":
                if not {"mean", "stdev"} <= set(d.params):
                    problems.append("{}: normal needs mean/stdev".format(
                        where))
            elif d.kind == "lognormal":
                if not {"mu", "sigma"} <= set(d.params):
                    problems.append("{}: lognormal needs mu/sigma".format(
                        where))
            elif d.kind == "date_range":
                if not {"start", "end"} <= set(d.params):
                    problems.append("{}: date_range needs start/end".format(
                        where))
            if not (0.0 <= f.nullable_rate <= 1.0):
                problems.append("{}: nullable_rate outside [0,1]".format(
                    where))

        for u in self.unstructured_fields:
            where = "unstructured field `{}`".format(u.name or "?")
            if not u.name:
                problems.append("unstructured field with empty name")
            elif u.name in names:
                problems.append("duplicate field name `{}`".format(u.name))
            names.add(u.name)
            if not u.target_elements:
                problems.append("{}: no target elements — nothing to "
                                "test".format(where))
            eids = set()
            for t in u.target_elements:
                ew = "{} element `{}`".format(where, t.element_id or "?")
                if not t.element_id:
                    problems.append("{}: element with empty id".format(
                        where))
                elif t.element_id in eids:
                    problems.append("duplicate element id `{}`".format(
                        t.element_id))
                eids.add(t.element_id)
                if len(t.phrasings) < 1:
                    problems.append("{}: needs at least 1 phrasing".format(
                        ew))
                if not (0.0 < t.density <= 1.0):
                    problems.append("{}: density outside (0,1]".format(ew))
                if t.difficulty not in DIFFICULTIES:
                    problems.append("{}: difficulty `{}` unknown".format(
                        ew, t.difficulty))
                problems.extend(_check_value_source(
                    t.value_source, t.phrasings, names, ew))
            for dtr in u.distractors:
                dw = "{} distractor `{}`".format(
                    where, dtr.distractor_id or "?")
                if dtr.distractor_id in eids:
                    problems.append(
                        "{}: id collides with a target element".format(dw))
                if not dtr.phrasings:
                    problems.append("{}: needs phrasings".format(dw))
                problems.extend(_check_value_source(
                    dtr.value_source, dtr.phrasings, names, dw))
            if (len(u.length_words) != 2
                    or u.length_words[0] > u.length_words[1]
                    or u.length_words[0] < 10):
                problems.append("{}: length_words must be [min>=10, "
                                "max>=min]".format(where))

        rule_fields = {f.name for f in self.structured_fields}
        for r in self.cross_field_rules:
            for side in (r.earlier, r.later):
                if side not in rule_fields:
                    problems.append(
                        "cross-field rule references unknown field "
                        "`{}`".format(side))
            if r.min_delta > r.max_delta:
                problems.append(
                    "cross-field rule {}<{}: min_delta > max_delta".format(
                        r.earlier, r.later))

        if self.corpus.size < 1:
            problems.append("corpus size must be >= 1")
        if not self.title:
            problems.append("spec needs a title")

        if problems:
            raise SpecError("\n".join(problems))


def _check_value_source(source: Any, phrasings: List[str],
                        field_names: set, where: str) -> List[str]:
    out: List[str] = []
    uses_value = any("{value}" in p for p in phrasings)
    if source is None:
        if uses_value:
            out.append("{}: phrasings use {{value}} but no "
                       "value_source".format(where))
        return out
    if isinstance(source, str):
        if source not in field_names:
            out.append("{}: value_source names unknown field "
                       "`{}`".format(where, source))
    elif isinstance(source, dict):
        if not source.get("choices"):
            out.append("{}: inline value_source needs choices".format(
                where))
    else:
        out.append("{}: value_source must be a field name or "
                   "{{'choices': [...]}}".format(where))
    if not uses_value:
        out.append("{}: value_source given but no phrasing uses "
                   "{{value}}".format(where))
    return out


## Library: LLM backends (Ollama / Anthropic / Bedrock) and the spec compiler

*Source of truth: `synthkit/compiler.py` — this cell is generated, not hand-edited.*


In [ ]:
"""SYNTH_V1 S1: backends and the SpecCompiler.

The LLMBackend protocol is the deployment seam: development runs
Ollama on the M3; Keck runs Bedrock or the Anthropic API. Every
LLM-touching stage takes a backend instance, so the move is a config
change, not a port.

The compiler is LLM-ASSISTED, not LLM-trusted: plain English goes
in, a draft DataSpec comes out, validation runs, and the human
reviews the JSON before anything generates. A compile that fails
validation returns the problems alongside the draft — the fix loop
is human-in-the-middle by design.

Python 3.8 compatible. Stdlib only (backends import their SDKs
lazily so the library works wherever at least one is available).
"""
from __future__ import annotations

import json
import logging
import re
from dataclasses import dataclass
from typing import Any, Optional, Tuple

pass  # intra-package import inlined above

log = logging.getLogger(__name__)


# ===================================================================
# Backends
# ===================================================================

class LLMBackend:
    """Protocol: complete(prompt, system, max_tokens, temperature)
    -> str. Adapters raise BackendError on hard failure."""

    name = "base"

    def complete(self, prompt: str, *, system: str = "",
                 max_tokens: int = 2000,
                 temperature: float = 0.3) -> str:
        raise NotImplementedError


class BackendError(RuntimeError):
    pass


class OllamaBackend(LLMBackend):
    """Local models via the Ollama HTTP API (development default)."""

    name = "ollama"

    def __init__(self, model: str = "mistral-small3.1",
                 host: str = "http://localhost:11434",
                 timeout_s: float = 300.0):
        self.model = model
        self.host = host.rstrip("/")
        self.timeout_s = timeout_s

    def complete(self, prompt: str, *, system: str = "",
                 max_tokens: int = 2000,
                 temperature: float = 0.3) -> str:
        import urllib.request
        body = json.dumps({
            "model": self.model,
            "prompt": prompt,
            "system": system,
            "stream": False,
            "options": {"temperature": temperature,
                        "num_predict": max_tokens},
        }).encode("utf-8")
        req = urllib.request.Request(
            self.host + "/api/generate", data=body,
            headers={"Content-Type": "application/json"})
        try:
            with urllib.request.urlopen(req,
                                        timeout=self.timeout_s) as r:
                return json.loads(r.read().decode("utf-8")).get(
                    "response", "")
        except Exception as e:
            raise BackendError("ollama call failed: {}".format(e))


class AnthropicBackend(LLMBackend):
    """Anthropic API (lazy SDK import)."""

    name = "anthropic"

    def __init__(self, model: str = "claude-sonnet-4-6",
                 api_key: Optional[str] = None):
        self.model = model
        self.api_key = api_key

    def complete(self, prompt: str, *, system: str = "",
                 max_tokens: int = 2000,
                 temperature: float = 0.3) -> str:
        try:
            import anthropic
        except ImportError:
            raise BackendError("anthropic SDK not installed")
        client = anthropic.Anthropic(api_key=self.api_key) \
            if self.api_key else anthropic.Anthropic()
        try:
            msg = client.messages.create(
                model=self.model,
                max_tokens=max_tokens,
                temperature=temperature,
                system=system or None,
                messages=[{"role": "user", "content": prompt}],
            )
            return "".join(
                b.text for b in msg.content
                if getattr(b, "type", "") == "text")
        except Exception as e:
            raise BackendError("anthropic call failed: {}".format(e))


class HFLocalBackend(LLMBackend):
    """IN-PROCESS open-source model via Hugging Face transformers —
    generation happens inside the notebook's own Python process, no
    external server or service.

        pip install transformers torch accelerate

    Weights download from the Hugging Face hub on first use (or pass
    a local path / S3-synced directory as model_id for air-gapped
    environments). Small instruct models are the sweet spot for
    synthkit's short verified renders:

        HFLocalBackend("Qwen/Qwen2.5-1.5B-Instruct")   # ~3GB
        HFLocalBackend("Qwen/Qwen2.5-0.5B-Instruct")   # ~1GB, CPU-ok

    The renderer's verifier + retry + fallback wrap this like any
    backend: a weak model degrades measurably, never breaks the
    corpus. `pipeline` is injectable for tests.
    """

    def __init__(self, model_id: str = "Qwen/Qwen2.5-1.5B-Instruct",
                 device_map: str = "auto",
                 pipeline: Any = None):
        self.model_id = model_id
        self.name = "hf-local/" + model_id
        self._device_map = device_map
        self._pipe = pipeline

    def _ensure_pipe(self) -> Any:
        if self._pipe is None:
            try:
                from transformers import pipeline as hf_pipeline
            except ImportError:
                raise BackendError(
                    "transformers is not installed — run: pip "
                    "install transformers torch accelerate")
            try:
                self._pipe = hf_pipeline(
                    "text-generation", model=self.model_id,
                    device_map=self._device_map)
            except Exception as e:
                raise BackendError(
                    "could not load {}: {}".format(self.model_id, e))
        return self._pipe

    def complete(self, prompt: str, *, system: str = "",
                 max_tokens: int = 2000,
                 temperature: float = 0.3) -> str:
        pipe = self._ensure_pipe()
        messages = []
        if system:
            messages.append({"role": "system", "content": system})
        messages.append({"role": "user", "content": prompt})
        kwargs = {"max_new_tokens": max_tokens,
                  "do_sample": temperature > 0,
                  "temperature": max(temperature, 0.01),
                  "return_full_text": False}
        try:
            out = pipe(messages, **kwargs)
        except (TypeError, ValueError):
            text_in = (system + "\n\n" + prompt) if system else prompt
            try:
                out = pipe(text_in, **kwargs)
            except Exception as e:
                raise BackendError("hf-local generation failed: "
                                   "{}".format(e))
        except Exception as e:
            raise BackendError("hf-local generation failed: "
                               "{}".format(e))
        try:
            text = out[0]["generated_text"]
            if isinstance(text, list):
                text = text[-1].get("content", "")
            return str(text)
        except (KeyError, IndexError, TypeError, AttributeError) as e:
            raise BackendError(
                "hf-local returned an unexpected shape: {}".format(e))


class BedrockBackend(LLMBackend):
    """AWS Bedrock via boto3 (the Keck deployment path).
    Anthropic-on-Bedrock message format."""

    name = "bedrock"

    def __init__(self, model_id: str, region: str = "us-west-2"):
        self.model_id = model_id
        self.region = region

    def complete(self, prompt: str, *, system: str = "",
                 max_tokens: int = 2000,
                 temperature: float = 0.3) -> str:
        try:
            import boto3
        except ImportError:
            raise BackendError("boto3 not installed")
        client = boto3.client("bedrock-runtime",
                              region_name=self.region)
        body = {
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": max_tokens,
            "temperature": temperature,
            "messages": [{"role": "user", "content": prompt}],
        }
        if system:
            body["system"] = system
        try:
            resp = client.invoke_model(
                modelId=self.model_id,
                body=json.dumps(body))
            parsed = json.loads(resp["body"].read())
            return "".join(
                b.get("text", "") for b in parsed.get("content", []))
        except Exception as e:
            raise BackendError("bedrock call failed: {}".format(e))


# ===================================================================
# The compiler
# ===================================================================

_COMPILER_SYSTEM = """You compile a plain-English description of a \
synthetic dataset into a strict JSON DataSpec. Output ONLY the JSON \
object, no prose, no markdown fences, starting with '{'.

Schema (all keys required unless noted):
{
  "title": "short name for the dataset",
  "structured_fields": [
    {"name": "field_name",
     "ftype": "int|float|str|date|categorical|id|person_name",
     "distribution": {"kind": "uniform|normal|lognormal|categorical|date_range|sequence",
                      "params": { ... kind-specific ... }},
     "nullable_rate": 0.0}
  ],
  "unstructured_fields": [
    {"name": "field_name", "note_type": "what kind of note this is",
     "target_elements": [
       {"element_id": "snake_case_id",
        "description": "the fact an extractor should find",
        "phrasings": ["two or more surface forms, use {value} where a sampled value goes"],
        "density": 0.8, "difficulty": "easy|medium|hard",
        "value_source": "structured_field_name OR {\\"choices\\": [..]} OR null"}
     ],
     "distractors": [
       {"distractor_id": "snake_case_id",
        "description": "plausible near-miss that should NOT be extracted",
        "phrasings": ["..."], "density": 0.5, "value_source": null}
     ],
     "style": {"personas": ["..."], "verbosity": ["terse","moderate","verbose"],
               "abbreviation": ["none","moderate","heavy"]},
     "length_words": [80, 220]}
  ],
  "cross_field_rules": [
    {"earlier": "field_a", "later": "field_b",
     "min_delta": 0, "max_delta": 30}
  ],
  "corpus": {"size": 50, "master_seed": 20260715}
}

Distribution params: uniform {"low","high"}; normal {"mean","stdev",
optional "min","max"}; lognormal {"mu","sigma",optional "min","max"};
categorical {"choices",optional "weights"}; date_range {"start","end"
as YYYY-MM-DD}; sequence {"prefix","start"} for ids.

Rules:
- Every target element gets >=2 phrasings with genuinely different
  surface structure.
- Include distractors: plausible near-misses of the targets.
- densities in (0,1]; difficulties spread across easy/medium/hard.
- NEVER invent real-sounding institutions, patients, or providers;
  values are generic and synthetic.
- Sizes and seeds: honor the user's numbers; default size 50."""


@dataclass
class CompileResult:
    spec: Optional[DataSpec]
    raw_json: str
    problems: Optional[str]      # None when the spec validates

    @property
    def ok(self) -> bool:
        return self.spec is not None and self.problems is None


def _extract_json(text: str) -> Optional[str]:
    if not text:
        return None
    m = re.search(r"\{.*\}", text, re.DOTALL)
    return m.group(0) if m else None


def compile_spec(description: str, backend: LLMBackend,
                 retries: int = 1) -> CompileResult:
    """Plain English -> validated DataSpec (or draft + problems).
    On validation failure, one repair round feeds the problems back
    to the model; after that, the human takes over — by design."""
    prompt = ("Dataset description:\n\n{}\n\n=== END DESCRIPTION ===\n"
              "Now output ONLY the DataSpec JSON object, starting "
              "with '{{'.".format(description.strip()))
    last_raw = ""
    last_problems = "no output produced"
    for attempt in range(retries + 1):
        raw = backend.complete(prompt, system=_COMPILER_SYSTEM,
                               max_tokens=3000, temperature=0.3)
        json_str = _extract_json(raw)
        if json_str is None:
            last_raw, last_problems = raw or "", "no JSON object in output"
        else:
            last_raw = json_str
            try:
                spec = DataSpec.from_json(json_str)
            except (json.JSONDecodeError, TypeError, KeyError) as e:
                last_problems = "JSON did not fit the schema: {}".format(e)
            else:
                try:
                    spec.validate()
                    return CompileResult(spec=spec, raw_json=json_str,
                                         problems=None)
                except SpecError as e:
                    last_problems = str(e)
                    log.info("compile attempt %d: %d validation "
                             "problem(s)", attempt + 1,
                             last_problems.count("\n") + 1)
        if attempt < retries:
            prompt = (
                "Dataset description:\n\n{}\n\n"
                "Your previous DataSpec had these problems:\n{}\n\n"
                "Output the CORRECTED DataSpec JSON only, starting "
                "with '{{'.".format(description.strip(), last_problems)
            )
    try:
        draft = DataSpec.from_json(last_raw)
    except Exception:
        draft = None
    return CompileResult(spec=draft, raw_json=last_raw,
                         problems=last_problems)


# ===================================================================
# Table compilation — same law as documents: LLM-assisted, never
# LLM-trusted. Draft -> total validation -> problems-fed repair
# round(s) -> human review. Returns CompileResult; `spec` holds a
# TableSpec when parseable (validated when ok).
# ===================================================================

_TABLE_COMPILER_SYSTEM = (
    "You translate plain-English descriptions of tabular datasets "
    "into synthkit TableSpec JSON. Respond ONLY with a JSON "
    "object, no prose, no markdown fences.\n\n"
    "Schema:\n"
    '{"title": str, "rows": int, "master_seed": int, '
    '"duplicate_rate": float 0-0.5,\n'
    ' "columns": [{"name": str, '
    '"ctype": "int|float|category|str_id|person_name|date|bool", '
    '"distribution": {"kind": ...}, "mess": {...}}],\n'
    ' "rules": [...]}\n\n'
    "Distribution kinds: uniform{min,max} normal{mean,std,min?,"
    "max?} lognormal{mu,sigma,min?,max?} beta{alpha,beta,scale?} "
    'categorical{choices,[weights]} date_range{start,end ISO} '
    "sequence{prefix,start} bernoulli{p} "
    "mixture{components:[dists],weights}.\n"
    "Mess rates (per column, all optional, 0-1): missing_rate, "
    "typo_rate, format_rate, outlier_rate (+outlier_factor), "
    "case_rate, space_rate, wrong_rate (format-valid wrong "
    "values a cleaner must DETECT).\n"
    "Rules (top-level `rules` array, NEVER inside a column's "
    "distribution): {kind:date_after, earlier, later, min_days, "
    "max_days} for date ordering — or replace min/max_days with "
    "days_from: <numeric column> when the gap should equal that "
    "column's value (e.g. discharge = admission + los_days). "
    "Rule-produced columns may omit `distribution` entirely. "
    "wrong_rate applies to dates, numbers, categories, bools and "
    "names — never str_id; {kind:derived, target, source, "
    "factor, noise_sigma?} for numeric correlation — both ends "
    "numeric; factor carries the SCALE (dollars per day etc), "
    "noise_sigma is lognormal shape in [0, 3] where 0.15 means "
    "about ±15% scatter.\n"
    "Generated labels (top-level `outcomes` array, NEVER a bool "
    "column with rules): {\"name\": str, \"kind\": "
    "\"logistic\", \"intercept\": float, \"coefficients\": "
    "{column: weight, \"column=value\": weight}} — numeric/bool "
    "columns by name, categories via indicator "
    "\"column=value\". Tune intercept for the requested "
    "prevalence (more negative = rarer), and when the "
    "description states a rate ('around 15-20 percent') also "
    "set \"target_prevalence\": [0.15, 0.20] so the linter "
    "can enforce it. For CONTINUOUS generated targets use "
    "kind \"linear\" with intercept, coefficients, and "
    "noise_sigma (the irreducible error), plus optional "
    "target_range [lo, hi] for the expected mean.\n"
    "Choose sensible values for anything unspecified; keep rows "
    "<= 500 unless asked."
)


def compile_table_spec(description: str, backend: LLMBackend,
                       retries: int = 1) -> CompileResult:
    """Plain English -> validated TableSpec (or draft + problems).
    Mirrors compile_spec exactly: one repair round fed the full
    problem list, then the human takes over."""
    pass  # intra-package import inlined above
    prompt = ("Dataset description:\n\n{}\n\n"
              "=== END DESCRIPTION ===\n"
              "Now output ONLY the TableSpec JSON object, starting "
              "with '{{'.".format(description.strip()))
    last_raw = ""
    last_problems = "no output produced"
    for attempt in range(retries + 1):
        raw = backend.complete(prompt,
                               system=_TABLE_COMPILER_SYSTEM,
                               max_tokens=3000, temperature=0.3)
        json_str = _extract_json(raw)
        if json_str is None:
            last_raw = raw or ""
            last_problems = "no JSON object in output"
        else:
            last_raw = json_str
            try:
                spec = TableSpec.from_json(json_str)
            except (json.JSONDecodeError, TypeError, KeyError) as e:
                last_problems = ("JSON did not fit the schema: {}"
                                 .format(e))
            else:
                try:
                    spec.validate()
                    return CompileResult(spec=spec,
                                         raw_json=json_str,
                                         problems=None)
                except TableSpecError as e:
                    last_problems = str(e)
        if attempt < retries:
            prompt = (
                "Dataset description:\n\n{}\n\n"
                "Your previous TableSpec had these problems:\n{}"
                "\n\nOutput the CORRECTED TableSpec JSON only, "
                "starting with '{{'.".format(
                    description.strip(), last_problems)
            )
    try:
        draft = TableSpec.from_json(last_raw)
    except Exception:
        draft = None
    return CompileResult(spec=draft, raw_json=last_raw,
                         problems=last_problems)


## Library: The Planner — deterministic blueprints, ground truth first

*Source of truth: `synthkit/planner.py` — this cell is generated, not hand-edited.*


In [ ]:
"""SYNTH_V1 S2: the Planner — spec in, blueprints out, seeded.

One blueprint per document: which target elements it contains (per
their densities), the sampled values, the drawn style, which
distractors are present, and every structured field's value. The
blueprint IS the ground truth — labels precede the data, so no
labeling step ever exists.

Determinism contract: the same spec + master_seed produces the same
blueprints byte for byte, on any machine, forever. Per-document RNGs
derive from (master_seed, doc_index) so corpus size changes never
reshuffle existing documents.

Cross-field rules are enforced CONSTRUCTIVELY: the planner samples
the `earlier` field and a delta, never rejection-samples — a spec
that validates always plans.

Python 3.8 compatible. Stdlib only.
"""
from __future__ import annotations

import hashlib
import json
import math
import random
from dataclasses import asdict, dataclass, field
from datetime import date, timedelta
from typing import Any, Dict, List, Optional

pass  # intra-package import inlined above

FIRST_NAMES = (
    "Avery", "Jordan", "Riley", "Quinn", "Morgan", "Casey", "Rowan",
    "Skyler", "Emerson", "Hayden", "Reese", "Dakota", "Finley",
    "Sage", "Marlowe", "Ellis",
)
LAST_NAMES = (
    "Calloway", "Mercer", "Ashford", "Brennan", "Voss", "Hale",
    "Winslow", "Marsh", "Keating", "Solano", "Iverson", "Trent",
    "Fontaine", "Barlow", "Quimby", "Renner",
)


@dataclass
class PlannedElement:
    element_id: str
    phrasing: str            # chosen phrasing with {value} resolved
    value: Optional[str]     # the resolved value ('' family), if any
    difficulty: str


@dataclass
class PlannedDistractor:
    distractor_id: str
    phrasing: str
    value: Optional[str]


@dataclass
class PlannedNote:
    field_name: str
    note_type: str
    style: Dict[str, str]            # axis -> drawn value
    length_words: int
    elements: List[PlannedElement] = field(default_factory=list)
    distractors: List[PlannedDistractor] = field(default_factory=list)


@dataclass
class Blueprint:
    doc_id: str
    doc_index: int
    seed: int
    structured: Dict[str, Any] = field(default_factory=dict)
    notes: List[PlannedNote] = field(default_factory=list)

    def to_json(self) -> str:
        return json.dumps(asdict(self), indent=2, ensure_ascii=False,
                          default=str)


def _doc_seed(master_seed: int, doc_index: int) -> int:
    """Stable per-document seed independent of corpus size."""
    h = hashlib.sha256("{}:{}".format(master_seed, doc_index)
                       .encode("utf-8")).hexdigest()
    return int(h[:12], 16)


def _sample_distribution(rng: random.Random, ftype: str,
                         d: Distribution) -> Any:
    p = d.params
    if d.kind == "uniform":
        v = rng.uniform(float(p["low"]), float(p["high"]))
        return int(round(v)) if ftype == "int" else round(v, 3)
    if d.kind == "normal":
        v = rng.gauss(float(p["mean"]), float(p["stdev"]))
        v = _clamp(v, p.get("min"), p.get("max"))
        return int(round(v)) if ftype == "int" else round(v, 3)
    if d.kind == "lognormal":
        v = rng.lognormvariate(float(p["mu"]), float(p["sigma"]))
        v = _clamp(v, p.get("min"), p.get("max"))
        return int(round(v)) if ftype == "int" else round(v, 3)
    if d.kind == "categorical":
        choices = p["choices"]
        weights = p.get("weights")
        return rng.choices(choices, weights=weights, k=1)[0]
    if d.kind == "date_range":
        start = date.fromisoformat(p["start"])
        end = date.fromisoformat(p["end"])
        span = max((end - start).days, 0)
        return (start + timedelta(days=rng.randint(0, span))).isoformat()
    if d.kind == "sequence":
        # Resolved by the caller (needs doc_index, not randomness).
        return None
    raise SpecError("unsupported distribution kind: {}".format(d.kind))


def _clamp(v: float, lo: Optional[Any], hi: Optional[Any]) -> float:
    if lo is not None:
        v = max(float(lo), v)
    if hi is not None:
        v = min(float(hi), v)
    return v


def _person_name(rng: random.Random) -> str:
    return "{} {}".format(rng.choice(FIRST_NAMES),
                          rng.choice(LAST_NAMES))


def _resolve_value(rng: random.Random, source: Any,
                   structured: Dict[str, Any]) -> Optional[str]:
    if source is None:
        return None
    if isinstance(source, str):
        return str(structured.get(source, ""))
    if isinstance(source, dict):
        return str(rng.choice(source["choices"]))
    return None


def plan_document(spec: DataSpec, doc_index: int) -> Blueprint:
    """One document's blueprint, fully determined by (spec, index)."""
    seed = _doc_seed(spec.corpus.master_seed, doc_index)
    rng = random.Random(seed)
    bp = Blueprint(
        doc_id="doc_{:05d}".format(doc_index),
        doc_index=doc_index,
        seed=seed,
    )

    # ---- structured fields (rule-constrained pairs handled after) ----
    ruled_later = {r.later: r for r in spec.cross_field_rules}
    for f in spec.structured_fields:
        if f.name in ruled_later:
            continue
        if f.nullable_rate and rng.random() < f.nullable_rate:
            bp.structured[f.name] = None
            continue
        if f.distribution.kind == "sequence":
            p = f.distribution.params
            bp.structured[f.name] = "{}{}".format(
                p.get("prefix", ""), int(p.get("start", 1)) + doc_index)
        elif f.ftype == "person_name":
            bp.structured[f.name] = _person_name(rng)
        else:
            bp.structured[f.name] = _sample_distribution(
                rng, f.ftype, f.distribution)

    for r in spec.cross_field_rules:
        earlier_val = bp.structured.get(r.earlier)
        f = next((x for x in spec.structured_fields
                  if x.name == r.later), None)
        if f is None or earlier_val is None:
            continue
        delta = rng.uniform(r.min_delta, r.max_delta)
        if f.ftype == "date":
            base = date.fromisoformat(str(earlier_val))
            bp.structured[f.name] = (
                base + timedelta(days=int(math.ceil(delta)))
            ).isoformat()
        else:
            v = float(earlier_val) + delta
            bp.structured[f.name] = (
                int(round(v)) if f.ftype == "int" else round(v, 3))

    # ---- unstructured notes ----
    for u in spec.unstructured_fields:
        note = PlannedNote(
            field_name=u.name,
            note_type=u.note_type,
            style={
                "persona": rng.choice(u.style.personas),
                "verbosity": rng.choice(u.style.verbosity),
                "abbreviation": rng.choice(u.style.abbreviation),
            },
            length_words=rng.randint(u.length_words[0],
                                     u.length_words[1]),
        )
        for t in u.target_elements:
            if rng.random() >= t.density:
                continue
            value = _resolve_value(rng, t.value_source, bp.structured)
            phrasing = rng.choice(t.phrasings)
            if value is not None:
                phrasing = phrasing.replace("{value}", value)
            note.elements.append(PlannedElement(
                element_id=t.element_id,
                phrasing=phrasing,
                value=value,
                difficulty=t.difficulty,
            ))
        for dtr in u.distractors:
            if rng.random() >= dtr.density:
                continue
            value = _resolve_value(rng, dtr.value_source, bp.structured)
            phrasing = rng.choice(dtr.phrasings)
            if value is not None:
                phrasing = phrasing.replace("{value}", value)
            note.distractors.append(PlannedDistractor(
                distractor_id=dtr.distractor_id,
                phrasing=phrasing,
                value=value,
            ))
        bp.notes.append(note)

    return bp


def plan_corpus(spec: DataSpec) -> List[Blueprint]:
    """Validates, then plans every document. Deterministic."""
    spec.validate()
    return [plan_document(spec, i) for i in range(spec.corpus.size)]


def corpus_stats(blueprints: List[Blueprint]) -> Dict[str, Any]:
    """Planned-corpus telemetry: realized element densities and style
    distribution — the pre-flight check that the corpus you are about
    to render actually exercises what the spec intended."""
    n = len(blueprints) or 1
    element_counts: Dict[str, int] = {}
    distractor_counts: Dict[str, int] = {}
    style_counts: Dict[str, Dict[str, int]] = {}
    for bp in blueprints:
        for note in bp.notes:
            for el in note.elements:
                element_counts[el.element_id] = (
                    element_counts.get(el.element_id, 0) + 1)
            for d in note.distractors:
                distractor_counts[d.distractor_id] = (
                    distractor_counts.get(d.distractor_id, 0) + 1)
            for axis, val in note.style.items():
                style_counts.setdefault(axis, {})
                style_counts[axis][val] = (
                    style_counts[axis].get(val, 0) + 1)
    return {
        "documents": len(blueprints),
        "element_density": {
            k: round(v / n, 3) for k, v in sorted(element_counts.items())
        },
        "distractor_density": {
            k: round(v / n, 3)
            for k, v in sorted(distractor_counts.items())
        },
        "style_distribution": style_counts,
    }


## Library: The Renderer — verified generation with targeted retries and fallback

*Source of truth: `synthkit/renderer.py` — this cell is generated, not hand-edited.*


In [ ]:
"""SYNTH_V1 S3: the Renderer — blueprints become documents, verified.

The LLM's one job here is prose: given a blueprint's planted
elements, distractors, style draws, and structured context, write a
natural note of the right register. Everything around that call is
code:

  VERIFIER (code, not LLM)
    - every planted element's needle (its value, or its phrasing
      when valueless) must appear in the text, normalized
    - every planted distractor's needle must appear (distractors are
      real content — the trap only works if it is present)
    - no OMITTED element may sneak in: for omitted elements with a
      choices pool, none of the pool values may appear — this is
      what keeps "absence is ground truth" true after an LLM has
      touched the data

  RETRY loop (default 3 attempts): failures are named verbatim in
  the retry prompt ("MUST include exactly: ...", "MUST NOT
  mention: ..."). Small window, explicit instruction — the only
  regime local models honor.

  FALLBACK: after retries, the deterministic stub renderer produces
  the document — valid by construction, flagged in the report. A
  corpus render therefore CANNOT fail; it can only report how much
  fallback it needed, which is itself renderer-quality telemetry.

Python 3.8 compatible. Stdlib only.
"""
from __future__ import annotations

import logging
import re
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple

pass  # intra-package import inlined above
pass  # intra-package import inlined above
pass  # intra-package import inlined above

log = logging.getLogger(__name__)

RENDER_ATTEMPTS = 3


def _norm(text: str) -> str:
    return re.sub(r"\s+", " ", str(text or "")).strip().lower()


def _needle(value: Optional[str], phrasing: str) -> str:
    return _norm(value) if value else _norm(phrasing)


# ===================================================================
# Verification
# ===================================================================

@dataclass
class VerifyResult:
    ok: bool
    missing: List[str] = field(default_factory=list)     # needles absent
    forbidden: List[str] = field(default_factory=list)   # leaked values


def _forbidden_values(spec_field: UnstructuredField,
                      note: PlannedNote) -> List[str]:
    """Pool values of OMITTED elements — their appearance would
    corrupt the absence side of the ground truth."""
    present = {el.element_id for el in note.elements}
    out: List[str] = []
    for t in spec_field.target_elements:
        if t.element_id in present:
            continue
        if isinstance(t.value_source, dict):
            out.extend(str(c) for c in
                       t.value_source.get("choices", []))
    # Values planted in THIS note are never forbidden even if they
    # also sit in an omitted element's pool.
    planted = {_norm(el.value) for el in note.elements if el.value}
    planted |= {_norm(d.value) for d in note.distractors if d.value}
    return [v for v in out if _norm(v) not in planted]


def verify_note(text: str, note: PlannedNote,
                spec_field: UnstructuredField) -> VerifyResult:
    body = _norm(text)
    missing: List[str] = []
    for el in note.elements:
        n = _needle(el.value, el.phrasing)
        if n and n not in body:
            missing.append(el.value or el.phrasing)
    for d in note.distractors:
        n = _needle(d.value, d.phrasing)
        if n and n not in body:
            missing.append(d.value or d.phrasing)
    forbidden = [v for v in _forbidden_values(spec_field, note)
                 if _norm(v) in body]
    return VerifyResult(ok=not missing and not forbidden,
                        missing=missing, forbidden=forbidden)


# ===================================================================
# Prompting
# ===================================================================

_RENDER_SYSTEM = (
    "You write realistic synthetic documents for testing information "
    "extraction systems. You will receive a note type, a style, "
    "structured context, and REQUIRED CONTENT items. Write ONE "
    "document only — no preamble, no markdown fences, no "
    "explanations.\n\n"
    "Hard rules:\n"
    "- Every REQUIRED CONTENT item's key text must appear VERBATIM "
    "(you may write naturally around it, but the exact text must be "
    "present).\n"
    "- Never mention anything listed under DO NOT MENTION.\n"
    "- Everything is synthetic; never invent real institutions or "
    "real people beyond the names given.\n"
    "- Match the requested persona, verbosity, abbreviation level, "
    "and approximate length."
)


def _style_line(note: PlannedNote) -> str:
    return ("persona: {persona}; verbosity: {verbosity}; "
            "abbreviations: {abbreviation}").format(**note.style)


def _render_prompt(bp: Blueprint, note: PlannedNote,
                   spec_field: UnstructuredField,
                   extra_missing: Optional[List[str]] = None,
                   extra_forbidden: Optional[List[str]] = None) -> str:
    required = [el.phrasing for el in note.elements]
    required += [d.phrasing for d in note.distractors]
    ctx = ", ".join("{}={}".format(k, v)
                    for k, v in sorted(bp.structured.items())
                    if v is not None)
    lines = [
        "Note type: {}".format(note.note_type),
        "Style: {}".format(_style_line(note)),
        "Approximate length: {} words".format(note.length_words),
        "Structured context (weave in naturally where sensible): "
        + ctx,
        "",
        "REQUIRED CONTENT (each key text VERBATIM):",
    ]
    lines += ["- {}".format(r) for r in required]
    forbidden = _forbidden_values(spec_field, note)
    if forbidden or extra_forbidden:
        lines.append("")
        lines.append("DO NOT MENTION:")
        lines += ["- {}".format(f)
                  for f in sorted(set(forbidden)
                                  | set(extra_forbidden or []))]
    if extra_missing:
        lines.append("")
        lines.append("YOUR PREVIOUS ATTEMPT OMITTED these — include "
                      "each VERBATIM this time:")
        lines += ["- {}".format(m) for m in extra_missing]
    if extra_forbidden:
        lines.append("")
        lines.append("YOUR PREVIOUS ATTEMPT MENTIONED these "
                      "forbidden items — write the document WITHOUT "
                      "any of them:")
        lines += ["- {}".format(f) for f in extra_forbidden]
    lines.append("")
    lines.append("=== WRITE THE DOCUMENT NOW ===")
    return "\n".join(lines)


# ===================================================================
# The stub renderer — deterministic, valid by construction
# ===================================================================

def render_stub(bp: Blueprint, note: PlannedNote) -> str:
    parts = []
    name = bp.structured.get("patient_name")
    ident = next((v for k, v in sorted(bp.structured.items())
                  if isinstance(v, str) and "-" in str(v)), None)
    parts.append("{} regarding {}{}.".format(
        note.note_type.capitalize(),
        name or "the patient",
        " ({})".format(ident) if ident else ""))
    parts.append("Documented by the {} ({}).".format(
        note.style.get("persona", "author"),
        _style_line(note)))
    for el in note.elements:
        parts.append(el.phrasing.rstrip(".") + ".")
    for d in note.distractors:
        parts.append(d.phrasing.rstrip(".") + ".")
    parts.append("Plan reviewed; reassess at next contact.")
    return " ".join(parts)


# ===================================================================
# Rendering
# ===================================================================

@dataclass
class RenderResult:
    doc_id: str
    field_name: str
    text: str
    attempts: int
    used_fallback: bool
    problems: List[str] = field(default_factory=list)


@dataclass
class RenderReport:
    documents: int = 0
    notes_rendered: int = 0
    first_try: int = 0
    retried: int = 0
    fallbacks: int = 0
    miss_counts: Dict[str, int] = field(default_factory=dict)

    def format_text(self) -> str:
        lines = [
            "RENDER REPORT: {} note(s) across {} document(s)".format(
                self.notes_rendered, self.documents),
            "  verified first try: {}".format(self.first_try),
            "  verified after retry: {}".format(self.retried),
            "  deterministic fallback: {}".format(self.fallbacks),
        ]
        if self.miss_counts:
            lines.append("  renderer misses by needle (pre-retry):")
            top = sorted(self.miss_counts.items(),
                         key=lambda kv: -kv[1])[:8]
            lines += ["    {:<44} {}".format(k[:44], v)
                      for k, v in top]
        return "\n".join(lines)


def render_note(bp: Blueprint, note: PlannedNote,
                spec_field: UnstructuredField,
                backend: LLMBackend,
                attempts: int = RENDER_ATTEMPTS,
                report: Optional[RenderReport] = None,
                ) -> RenderResult:
    """One note: render -> verify -> targeted retry -> fallback."""
    extra_missing: List[str] = []
    extra_forbidden: List[str] = []
    problems: List[str] = []
    for attempt in range(1, attempts + 1):
        prompt = _render_prompt(bp, note, spec_field,
                                extra_missing or None,
                                extra_forbidden or None)
        try:
            text = backend.complete(
                prompt, system=_RENDER_SYSTEM,
                max_tokens=max(400, note.length_words * 3),
                temperature=0.7 if attempt == 1 else 0.4,
            )
        except BackendError as e:
            problems.append("attempt {}: backend error: {}".format(
                attempt, e))
            continue
        v = verify_note(text or "", note, spec_field)
        if v.ok:
            return RenderResult(
                doc_id=bp.doc_id, field_name=note.field_name,
                text=text.strip(), attempts=attempt,
                used_fallback=False, problems=problems)
        problems.append("attempt {}: missing={} forbidden={}".format(
            attempt, v.missing, v.forbidden))
        if report is not None:
            for m in v.missing:
                report.miss_counts[m] = (
                    report.miss_counts.get(m, 0) + 1)
        extra_missing = v.missing
        extra_forbidden = v.forbidden
        log.info("S3: %s/%s attempt %d failed verification "
                 "(%d missing, %d forbidden)", bp.doc_id,
                 note.field_name, attempt, len(v.missing),
                 len(v.forbidden))
    return RenderResult(
        doc_id=bp.doc_id, field_name=note.field_name,
        text=render_stub(bp, note), attempts=attempts,
        used_fallback=True, problems=problems)


def render_corpus(spec: DataSpec, blueprints: List[Blueprint],
                  backend: LLMBackend,
                  attempts: int = RENDER_ATTEMPTS,
                  ) -> Tuple[Dict[str, str], RenderReport]:
    """Every blueprint's notes rendered and verified. Returns
    ({doc_id: text}, report). Multi-note documents concatenate with
    field headers. CANNOT fail — only degrade, measurably."""
    fields = {u.name: u for u in spec.unstructured_fields}
    report = RenderReport(documents=len(blueprints))
    documents: Dict[str, str] = {}
    for bp in blueprints:
        chunks: List[str] = []
        for note in bp.notes:
            spec_field = fields[note.field_name]
            r = render_note(bp, note, spec_field, backend,
                            attempts=attempts, report=report)
            report.notes_rendered += 1
            if r.used_fallback:
                report.fallbacks += 1
            elif r.attempts == 1:
                report.first_try += 1
            else:
                report.retried += 1
            if len(bp.notes) > 1:
                chunks.append("[{}]\n{}".format(note.field_name,
                                                r.text))
            else:
                chunks.append(r.text)
        documents[bp.doc_id] = "\n\n".join(chunks)
    return documents, report


## Library: Corpus I/O — the reproducible, integrity-checked run artifact

*Source of truth: `synthkit/corpus_io.py` — this cell is generated, not hand-edited.*


In [ ]:
"""SYNTH_V1 S3: corpus I/O — a run is a reproducible, auditable
object you can hand to a colleague.

Layout of corpus/<run_id>/:
    manifest.json    spec (verbatim), master seed, backend name,
                     render stats, sha256 of every artifact
    docs/<doc_id>.txt
    truth/<doc_id>.json      the blueprint — labels precede data
    render_report.json

load_corpus verifies every hash on the way in: a corpus that fails
integrity refuses to load, because an edited document silently
diverging from its blueprint would poison every evaluation after it.

Python 3.8 compatible. Stdlib only.
"""
from __future__ import annotations

import hashlib
import json
from dataclasses import asdict
from pathlib import Path
from typing import Any, Dict, List, Tuple

pass  # intra-package import inlined above
pass  # intra-package import inlined above
pass  # intra-package import inlined above


class CorpusIntegrityError(RuntimeError):
    pass


def _sha(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def write_corpus(run_dir: Path, spec: DataSpec,
                 blueprints: List[Blueprint],
                 documents: Dict[str, str],
                 report: RenderReport,
                 backend_name: str = "unknown") -> Path:
    run_dir = Path(run_dir)
    (run_dir / "docs").mkdir(parents=True, exist_ok=True)
    (run_dir / "truth").mkdir(parents=True, exist_ok=True)
    hashes: Dict[str, str] = {}
    for bp in blueprints:
        text = documents.get(bp.doc_id, "")
        doc_rel = "docs/{}.txt".format(bp.doc_id)
        truth_rel = "truth/{}.json".format(bp.doc_id)
        (run_dir / doc_rel).write_text(text, encoding="utf-8")
        truth_json = bp.to_json()
        (run_dir / truth_rel).write_text(truth_json, encoding="utf-8")
        hashes[doc_rel] = _sha(text)
        hashes[truth_rel] = _sha(truth_json)
    report_json = json.dumps(asdict(report), indent=2)
    (run_dir / "render_report.json").write_text(report_json,
                                                encoding="utf-8")
    manifest = {
        "synthkit_manifest": 1,
        "spec": json.loads(spec.to_json()),
        "master_seed": spec.corpus.master_seed,
        "backend": backend_name,
        "documents": len(blueprints),
        "render": {
            "first_try": report.first_try,
            "retried": report.retried,
            "fallbacks": report.fallbacks,
        },
        "hashes": hashes,
    }
    (run_dir / "manifest.json").write_text(
        json.dumps(manifest, indent=2, ensure_ascii=False),
        encoding="utf-8")
    return run_dir


def load_corpus(run_dir: Path,
                ) -> Tuple[DataSpec, List[Blueprint],
                           Dict[str, str], Dict[str, Any]]:
    """Returns (spec, blueprints, documents, manifest). Every hash
    verified; any mismatch raises CorpusIntegrityError naming the
    file."""
    run_dir = Path(run_dir)
    manifest = json.loads(
        (run_dir / "manifest.json").read_text(encoding="utf-8"))
    spec = DataSpec.from_json(json.dumps(manifest["spec"]))
    documents: Dict[str, str] = {}
    blueprints: List[Blueprint] = []
    bad: List[str] = []
    for rel, expected in sorted(manifest["hashes"].items()):
        path = run_dir / rel
        text = path.read_text(encoding="utf-8")
        if _sha(text) != expected:
            bad.append(rel)
            continue
        if rel.startswith("docs/"):
            documents[path.stem] = text
        else:
            raw = json.loads(text)
            blueprints.append(_blueprint_from_raw(raw))
    if bad:
        raise CorpusIntegrityError(
            "corpus integrity failed for: {}".format(", ".join(bad)))
    blueprints.sort(key=lambda b: b.doc_index)
    return spec, blueprints, documents, manifest


def _blueprint_from_raw(raw: Dict[str, Any]) -> Blueprint:
    pass  # intra-package import inlined above
    return Blueprint(
        doc_id=raw["doc_id"],
        doc_index=raw["doc_index"],
        seed=raw["seed"],
        structured=raw.get("structured", {}),
        notes=[
            PlannedNote(
                field_name=n["field_name"],
                note_type=n["note_type"],
                style=n["style"],
                length_words=n["length_words"],
                elements=[PlannedElement(**e)
                          for e in n.get("elements", [])],
                distractors=[PlannedDistractor(**d)
                             for d in n.get("distractors", [])],
            )
            for n in raw.get("notes", [])
        ],
    )


## Library: The Evaluator — exact, sliced scoring against planted truth

*Source of truth: `synthkit/evaluator.py` — this cell is generated, not hand-edited.*


In [ ]:
"""SYNTH_V1 S4: the Evaluator — outside model vs blueprint truth.

Because synthkit PLANTED every fact, evaluation is exact alignment,
not judgment: an extraction matches a blueprint element when the
planted VALUE appears in the extraction's text (normalized), and —
when the rule demands it — the extraction's category agrees. The
same mechanism makes distractors provable false positives: if the
model reports the discontinued medication's value under a
current-medication category, that is a counted FP, not an opinion.

Scoring:
  per element    planted / found / missed -> recall
  distractors    planted / falsely extracted -> fp_rate
  slices         recall by difficulty and by every style axis the
                 planner controlled (persona, verbosity,
                 abbreviation) — the report that says WHERE a model
                 breaks, which no real-data test can isolate.
  unmatched      extractions matching nothing are reported but not
                 penalized (models may extract beyond the spec).

The evaluator is pure: (blueprints, documents, extractions) in,
EvalReport out. Adapters supply the extractions; the bundled
FunctionExtractor wraps any callable for quick harnessing.

Python 3.8 compatible. Stdlib only.
"""
from __future__ import annotations

import json
import re
from dataclasses import asdict, dataclass, field
from typing import Any, Callable, Dict, List, Optional

pass  # intra-package import inlined above


# ===================================================================
# Extraction side
# ===================================================================

@dataclass
class Extraction:
    """One thing the outside model claims to have extracted.

    category  the model's own label for it ("current_medication",
              "medication", "allergy", ...) — free text, matched by
              substring against rule categories
    text      the extracted content
    """
    category: str
    text: str


class ExtractorAdapter:
    """Protocol: extract(doc_id, doc_text) -> List[Extraction].
    Wrap the outside model here (HTTP call, SDK, subprocess — the
    evaluator does not care)."""

    name = "base"

    def extract(self, doc_id: str, doc_text: str) -> List[Extraction]:
        raise NotImplementedError


class FunctionExtractor(ExtractorAdapter):
    """Adapter for any callable(doc_id, doc_text) -> [Extraction]."""

    def __init__(self, fn: Callable[[str, str], List[Extraction]],
                 name: str = "function"):
        self._fn = fn
        self.name = name

    def extract(self, doc_id: str, doc_text: str) -> List[Extraction]:
        return self._fn(doc_id, doc_text)


# ===================================================================
# Matching rules
# ===================================================================

@dataclass
class MatchRule:
    """How a planted element (or distractor) aligns with extractions.

    mode         "value"  — planted value appears in extraction text
                 "phrase" — planted phrasing appears (for elements
                            with no {value})
    categories   optional list of substrings; when given, only
                 extractions whose category contains one (case-
                 insensitive) can match. This is what turns a
                 distractor hit into a PROVABLE false positive: the
                 discontinued med extracted under a current-med
                 category.
    """
    mode: str = "value"
    categories: Optional[List[str]] = None


def _norm(text: str) -> str:
    return re.sub(r"\s+", " ", str(text or "")).strip().lower()


def _category_ok(rule: MatchRule, category: str) -> bool:
    if not rule.categories:
        return True
    cat = _norm(category)
    return any(_norm(c) in cat for c in rule.categories)


def _needle_for(rule: MatchRule, value: Optional[str],
                phrasing: str) -> str:
    if rule.mode == "phrase" or not value:
        return _norm(phrasing)
    return _norm(value)


# ===================================================================
# Report
# ===================================================================

@dataclass
class ElementScore:
    element_id: str
    planted: int = 0
    found: int = 0
    missed_docs: List[str] = field(default_factory=list)

    @property
    def recall(self) -> float:
        return self.found / self.planted if self.planted else 0.0


@dataclass
class DistractorScore:
    distractor_id: str
    planted: int = 0
    false_positives: int = 0
    fp_docs: List[str] = field(default_factory=list)

    @property
    def fp_rate(self) -> float:
        return (self.false_positives / self.planted
                if self.planted else 0.0)


@dataclass
class EvalReport:
    extractor_name: str
    documents: int
    elements: Dict[str, ElementScore]
    distractors: Dict[str, DistractorScore]
    recall_by_difficulty: Dict[str, Dict[str, float]]
    recall_by_style: Dict[str, Dict[str, Dict[str, float]]]
    unmatched_extractions: int
    total_extractions: int

    @property
    def overall_recall(self) -> float:
        planted = sum(e.planted for e in self.elements.values())
        found = sum(e.found for e in self.elements.values())
        return found / planted if planted else 0.0

    def to_json(self) -> str:
        raw = asdict(self)
        raw["overall_recall"] = round(self.overall_recall, 4)
        for eid, e in self.elements.items():
            raw["elements"][eid]["recall"] = round(
                self.elements[eid].recall, 4)
        for did in self.distractors:
            raw["distractors"][did]["fp_rate"] = round(
                self.distractors[did].fp_rate, 4)
        return json.dumps(raw, indent=2, ensure_ascii=False)

    def format_text(self) -> str:
        lines = [
            "EVALUATION: {} on {} document(s)".format(
                self.extractor_name, self.documents),
            "overall element recall: {:.1%}".format(
                self.overall_recall),
            "",
            "PER ELEMENT:",
        ]
        for eid, e in sorted(self.elements.items()):
            lines.append(
                "  {:<28} recall {:>6.1%}  ({}/{} planted{})".format(
                    eid, e.recall, e.found, e.planted,
                    "; missed: " + ", ".join(e.missed_docs[:4])
                    + ("..." if len(e.missed_docs) > 4 else "")
                    if e.missed_docs else "",
                ))
        if self.distractors:
            lines.append("")
            lines.append("DISTRACTORS (false-positive traps):")
            for did, d in sorted(self.distractors.items()):
                lines.append(
                    "  {:<28} fp rate {:>5.1%}  ({}/{} planted)".format(
                        did, d.fp_rate, d.false_positives, d.planted))
        lines.append("")
        lines.append("RECALL BY DIFFICULTY:")
        for diff, cell in sorted(self.recall_by_difficulty.items()):
            lines.append("  {:<8} {:>6.1%}  ({} planted)".format(
                diff, cell["recall"], int(cell["planted"])))
        for axis, values in sorted(self.recall_by_style.items()):
            lines.append("")
            lines.append("RECALL BY {}:".format(axis.upper()))
            for val, cell in sorted(values.items()):
                lines.append("  {:<12} {:>6.1%}  ({} planted)".format(
                    val, cell["recall"], int(cell["planted"])))
        lines.append("")
        lines.append("extractions: {} total, {} matched nothing "
                     "(reported, not penalized)".format(
                         self.total_extractions,
                         self.unmatched_extractions))
        return "\n".join(lines)


# ===================================================================
# The evaluation
# ===================================================================

def evaluate(
    blueprints: List[Blueprint],
    documents: Dict[str, str],
    extractor: ExtractorAdapter,
    rules: Optional[Dict[str, MatchRule]] = None,
) -> EvalReport:
    """Run the extractor over every document and align against the
    blueprints. `rules` maps element_id/distractor_id -> MatchRule
    (missing ids get the default value-match-any-category rule)."""
    rules = rules or {}

    def rule_for(key: str) -> MatchRule:
        return rules.get(key, MatchRule())

    elements: Dict[str, ElementScore] = {}
    distractors: Dict[str, DistractorScore] = {}
    diff_cells: Dict[str, List[int]] = {}
    style_cells: Dict[str, Dict[str, List[int]]] = {}
    unmatched = 0
    total = 0

    for bp in blueprints:
        text = documents.get(bp.doc_id)
        if text is None:
            continue
        extractions = extractor.extract(bp.doc_id, text)
        total += len(extractions)
        used = [False] * len(extractions)

        for note in bp.notes:
            for el in note.elements:
                score = elements.setdefault(
                    el.element_id, ElementScore(el.element_id))
                score.planted += 1
                rule = rule_for(el.element_id)
                needle = _needle_for(rule, el.value, el.phrasing)
                hit = False
                for i, ex in enumerate(extractions):
                    if not _category_ok(rule, ex.category):
                        continue
                    if needle and needle in _norm(ex.text):
                        hit = True
                        used[i] = True
                        break
                if hit:
                    score.found += 1
                else:
                    score.missed_docs.append(bp.doc_id)
                dc = diff_cells.setdefault(el.difficulty, [0, 0])
                dc[0] += 1
                dc[1] += 1 if hit else 0
                for axis, val in note.style.items():
                    ax = style_cells.setdefault(axis, {})
                    cell = ax.setdefault(val, [0, 0])
                    cell[0] += 1
                    cell[1] += 1 if hit else 0

            for dtr in note.distractors:
                dscore = distractors.setdefault(
                    dtr.distractor_id,
                    DistractorScore(dtr.distractor_id))
                dscore.planted += 1
                rule = rule_for(dtr.distractor_id)
                needle = _needle_for(rule, dtr.value, dtr.phrasing)
                for i, ex in enumerate(extractions):
                    if not _category_ok(rule, ex.category):
                        continue
                    if needle and needle in _norm(ex.text):
                        dscore.false_positives += 1
                        dscore.fp_docs.append(bp.doc_id)
                        used[i] = True
                        break

        unmatched += sum(1 for u in used if not u)

    return EvalReport(
        extractor_name=extractor.name,
        documents=len([b for b in blueprints
                       if b.doc_id in documents]),
        elements=elements,
        distractors=distractors,
        recall_by_difficulty={
            k: {"planted": v[0],
                "recall": (v[1] / v[0]) if v[0] else 0.0}
            for k, v in diff_cells.items()
        },
        recall_by_style={
            axis: {
                val: {"planted": c[0],
                      "recall": (c[1] / c[0]) if c[0] else 0.0}
                for val, c in vals.items()
            }
            for axis, vals in style_cells.items()
        },
        unmatched_extractions=unmatched,
        total_extractions=total,
    )


def resolve_eval_counts(report: EvalReport, path: str):
    """(k, n) behind extraction proportion metrics."""
    if path == "overall_recall":
        planted = sum(e.planted for e in report.elements.values())
        found = sum(e.found for e in report.elements.values())
        return (found, planted)
    parts = path.split(".")
    if parts[0] == "elements" and len(parts) == 3 \
            and parts[1] in report.elements \
            and parts[2] == "recall":
        sc = report.elements[parts[1]]
        return (sc.found, sc.planted)
    if parts[0] == "distractors" and len(parts) == 3 \
            and parts[1] in report.distractors \
            and parts[2] == "fp_rate":
        sc = report.distractors[parts[1]]
        return (sc.false_positives, sc.planted)
    return None


## Library: The Harness — hypotheses become measured experiments

*Source of truth: `synthkit/harness.py` — this cell is generated, not hand-edited.*


In [ ]:
"""SYNTH_V1 S5: the harness — hypotheses become experiments.

The rest of synthkit answers "how does model X behave on data with
properties Y?" The harness turns that into a verdict machine:

    Experiment = spec + extractor + conditions
    run_experiment -> ExperimentResult(passed, measured, finding)

A Condition is a metric path against a threshold —
    overall_recall >= 0.9
    elements.allergy_flag.recall >= 0.85
    distractors.discontinued_medication.fp_rate <= 0.05
    recall_by_style.verbosity.terse.recall >= 0.7
— resolved against the S4 report. All conditions must hold for the
experiment to pass, and the finding names every measurement either
way: the verdict carries its evidence.

This is the piece that plugs synthkit into any hypothesis-testing
loop (Mnemo's forge included, via a thin adapter on that side):
empirical measurement replacing LLM-judged opinion.

StubBackend renders deterministically (planted content, boring
prose) for dry runs and CI; swap in OllamaBackend for realistic
prose without touching the experiment.

Python 3.8 compatible. Stdlib only.
"""
from __future__ import annotations

import re
from dataclasses import dataclass, field
from typing import Dict, List, Optional

pass  # intra-package import inlined above
pass  # intra-package import inlined above
pass  # intra-package import inlined above
pass  # intra-package import inlined above
pass  # intra-package import inlined above

OPS = (">=", "<=", ">", "<", "==")


class StubBackend(LLMBackend):
    """Deterministic renderer backend: echoes every REQUIRED CONTENT
    item into plain prose. Valid by construction — the dry-run and
    CI workhorse."""

    name = "stub"

    def complete(self, prompt: str, *, system: str = "",
                 max_tokens: int = 2000,
                 temperature: float = 0.3) -> str:
        reqs: List[str] = []
        active = False
        for line in prompt.splitlines():
            if line.startswith("REQUIRED CONTENT"):
                active = True
                continue
            if active:
                if line.startswith("- "):
                    reqs.append(line[2:])
                elif not line.strip():
                    active = False
        return ("Documented at the bedside today. "
                + " Also noted, ".join(r.rstrip(".") for r in reqs)
                + ". Plan continues as discussed.")


@dataclass
class Condition:
    metric: str          # dotted path into the eval report
    op: str              # one of OPS
    threshold: float

    def holds(self, value: float) -> bool:
        if self.op == ">=":
            return value >= self.threshold
        if self.op == "<=":
            return value <= self.threshold
        if self.op == ">":
            return value > self.threshold
        if self.op == "<":
            return value < self.threshold
        return abs(value - self.threshold) < 1e-9

    def describe(self, value: float) -> str:
        return "{} = {:.3f} (required {} {:.3f}) -> {}".format(
            self.metric, value, self.op, self.threshold,
            "HOLDS" if self.holds(value) else "FAILS")

    @staticmethod
    def parse(text: str) -> "Condition":
        """'elements.x.recall >= 0.9' -> Condition."""
        m = re.match(r"\s*([\w.]+)\s*(>=|<=|>|<|==)\s*([\d.]+)\s*$",
                     text)
        if not m:
            raise ValueError("cannot parse condition: {}".format(text))
        return Condition(metric=m.group(1), op=m.group(2),
                         threshold=float(m.group(3)))


class MetricError(KeyError):
    pass


def resolve_metric(report: EvalReport, path: str) -> float:
    """Dotted-path lookup with the report's computed properties."""
    parts = path.split(".")
    try:
        if parts[0] == "overall_recall":
            return report.overall_recall
        if parts[0] == "elements":
            score = report.elements[parts[1]]
            return {"recall": score.recall,
                    "planted": float(score.planted),
                    "found": float(score.found)}[parts[2]]
        if parts[0] == "distractors":
            d = report.distractors[parts[1]]
            return {"fp_rate": d.fp_rate,
                    "planted": float(d.planted),
                    "false_positives": float(d.false_positives)
                    }[parts[2]]
        if parts[0] == "recall_by_difficulty":
            return float(report.recall_by_difficulty[parts[1]]
                         [parts[2]])
        if parts[0] == "recall_by_style":
            return float(report.recall_by_style[parts[1]][parts[2]]
                         [parts[3]])
        if parts[0] == "unmatched_extractions":
            return float(report.unmatched_extractions)
    except (KeyError, IndexError):
        raise MetricError(
            "metric not present in this report: {}".format(path))
    raise MetricError("unknown metric family: {}".format(path))


@dataclass
class Experiment:
    name: str
    spec: DataSpec
    extractor: ExtractorAdapter
    conditions: List[Condition]
    rules: Optional[Dict[str, MatchRule]] = None
    backend: Optional[LLMBackend] = None    # default StubBackend
    render_attempts: int = 3


@dataclass
class ExperimentResult:
    name: str
    passed: bool
    measured: Dict[str, float]
    failed_conditions: List[str]
    eval_report: EvalReport
    render_report: RenderReport
    documents: int

    def finding(self) -> str:
        lines = [
            "Experiment '{}' on {} synthetic document(s) "
            "({} render fallback(s)): {}.".format(
                self.name, self.documents,
                self.render_report.fallbacks,
                "PASSED" if self.passed else "FAILED"),
        ]
        for cond_desc in self.measured_descriptions:
            lines.append("  " + cond_desc)
        return "\n".join(lines)

    measured_descriptions: List[str] = field(default_factory=list)


def run_experiment(exp: Experiment) -> ExperimentResult:
    """Plan -> render (verified) -> evaluate -> verdict. Deterministic
    for a given spec/seed/backend."""
    exp.spec.validate()
    blueprints = plan_corpus(exp.spec)
    backend = exp.backend or StubBackend()
    documents, render_report = render_corpus(
        exp.spec, blueprints, backend,
        attempts=exp.render_attempts)
    report = evaluate(blueprints, documents, exp.extractor,
                      exp.rules)
    measured: Dict[str, float] = {}
    descriptions: List[str] = []
    failed: List[str] = []
    for cond in exp.conditions:
        value = resolve_metric(report, cond.metric)
        measured[cond.metric] = round(value, 4)
        descriptions.append(cond.describe(value))
        if not cond.holds(value):
            failed.append(cond.metric)
    result = ExperimentResult(
        name=exp.name,
        passed=not failed,
        measured=measured,
        failed_conditions=failed,
        eval_report=report,
        render_report=render_report,
        documents=len(blueprints),
    )
    result.measured_descriptions = descriptions
    return result


## Library: TableSpec — arbitrary tabular datasets with planted mess

*Source of truth: `synthkit/tablespec.py` — this cell is generated, not hand-edited.*


In [ ]:
"""SYNTH_A1: TableSpec — arbitrary tabular datasets, mess included.

The tabular half of synthkit: the user declares columns (type +
distribution) and MESS POLICIES (missingness, typos, format drift,
outliers, case drift, whitespace, duplicate rows). The planner
generates the CLEAN truth first, then applies mess deterministically
and records every corruption in a ledger — so "can vendor X clean
this?" becomes an exact, cell-level, sliced measurement instead of
an opinion.

Column types: int, float, category, str_id, person_name, date, bool
Distributions (per type where sensible):
    uniform      {min,max}
    normal       {mean,std,min?,max?}
    lognormal    {mu,sigma,min?,max?}
    beta         {alpha,beta,scale?}          (floats in [0,scale])
    categorical  {choices,[weights]}          (weights optional)
    date_range   {start,end}                  (ISO dates)
    sequence     {prefix,start}               (ids)
    bernoulli    {p}                          (bools)

Mess (per column, all rates in [0,1], all OFF by default):
    missing_rate    cell replaced by a missing token
    typo_rate       strings: one seeded char swap/drop/double
    format_rate     dates -> mixed formats; numbers -> separators
    outlier_rate    numerics scaled by outlier_factor
    case_rate       strings: upper/lower/title drift
    space_rate      leading/trailing whitespace
Table-level:
    duplicate_rate  fraction of rows duplicated (appended, marked
                    in the ledger)

Validation is total (every problem reported at once); JSON
round-trips exactly. Python 3.8 compatible. Stdlib only.
"""
from __future__ import annotations

import json
from dataclasses import asdict, dataclass, field
from typing import Any, Dict, List, Optional

MISSING_TOKENS = ["", "NULL", "N/A", "?"]

COLUMN_TYPES = ("int", "float", "category", "str_id",
                "person_name", "date", "bool")
DIST_KINDS = ("uniform", "normal", "lognormal", "beta",
              "categorical", "date_range", "sequence", "bernoulli",
              "mixture")

RULE_KINDS = ("date_after", "derived")

_TYPE_DISTS = {
    "int": {"uniform", "normal", "lognormal", "sequence",
            "mixture"},
    "float": {"uniform", "normal", "lognormal", "beta",
              "mixture"},
    "category": {"categorical"},
    "str_id": {"sequence"},
    "person_name": {"categorical"},   # ignored; names synthesized
    "date": {"date_range"},
    "bool": {"bernoulli"},
}


class TableSpecError(ValueError):
    pass


@dataclass
class ColumnMess:
    missing_rate: float = 0.0
    missing_tokens: List[str] = field(
        default_factory=lambda: list(MISSING_TOKENS))
    typo_rate: float = 0.0
    format_rate: float = 0.0
    outlier_rate: float = 0.0
    outlier_factor: float = 10.0
    case_rate: float = 0.0
    space_rate: float = 0.0
    # The plausible-lie tier: the cell stays FORMAT-VALID but
    # carries a wrong value (date shifted, digits transposed,
    # category swapped, bool flipped). Normalizers cannot fix
    # these; real data-quality tools must DETECT them.
    wrong_rate: float = 0.0

    def any_active(self) -> bool:
        return any(r > 0 for r in (
            self.missing_rate, self.typo_rate, self.format_rate,
            self.outlier_rate, self.case_rate, self.space_rate,
            self.wrong_rate))


@dataclass
class ColumnSpec:
    name: str
    ctype: str
    distribution: Dict[str, Any] = field(default_factory=dict)
    mess: ColumnMess = field(default_factory=ColumnMess)

    def dist_kind(self) -> str:
        return str(self.distribution.get("kind", ""))


@dataclass
class TableSpec:
    title: str
    columns: List[ColumnSpec]
    rows: int = 100
    master_seed: int = 42
    duplicate_rate: float = 0.0
    # Cross-column rules, applied in order over generated columns:
    #  {"kind":"date_after","earlier":"admit","later":"discharge",
    #   "min_days":1,"max_days":30}
    #  {"kind":"derived","target":"total_cost","source":"los_days",
    #   "factor":1200,"noise_sigma":0.15}  (multiplicative noise)
    rules: List[Dict[str, Any]] = field(default_factory=list)
    # Generated labels with PLANTED SIGNAL: a logistic model over
    # this table's own columns. Every row gets a true probability,
    # which makes ceiling metrics computable.
    #  {"name":"readmitted","kind":"logistic","intercept":-2.0,
    #   "coefficients":{"age":0.03,"los_days":0.1,
    #                   "department=oncology":0.8,"active":-0.4}}
    # Coefficient keys: numeric column, bool column (0/1), or
    # "column=value" indicator for categoricals.
    outcomes: List[Dict[str, Any]] = field(default_factory=list)

    # ---------------- validation (total) ----------------
    def validate(self) -> None:
        problems: List[str] = []
        if not self.title.strip():
            problems.append("table title is required")
        if self.rows < 1:
            problems.append("rows must be >= 1")
        if not (0.0 <= self.duplicate_rate <= 0.5):
            problems.append("duplicate_rate must be in [0, 0.5]")
        if not self.columns:
            problems.append("at least one column is required")
        rule_targets = set()
        for rule in self.rules:
            if rule.get("kind") == "date_after":
                rule_targets.add(rule.get("later"))
            elif rule.get("kind") == "derived":
                rule_targets.add(rule.get("target"))
        seen = set()
        for col in self.columns:
            tag = "column `{}`".format(col.name or "?")
            if not col.name.strip():
                problems.append("a column is missing a name")
            elif col.name in seen:
                problems.append("{}: duplicate name".format(tag))
            seen.add(col.name)
            if col.ctype not in COLUMN_TYPES:
                problems.append("{}: unknown type `{}`".format(
                    tag, col.ctype))
                continue
            kind = col.dist_kind()
            if col.ctype == "person_name":
                pass  # synthesized; distribution optional
            elif not col.distribution:
                # Rule-produced columns need no distribution —
                # requiring a throwaway one made a live compile
                # invent placeholders.
                if col.name not in rule_targets:
                    problems.append(
                        "{}: needs a distribution (or a rule "
                        "that produces it)".format(tag))
            elif kind in DIST_KINDS and kind not in                     _TYPE_DISTS.get(col.ctype, set())                     and col.name in rule_targets:
                problems.append(
                    "{}: distribution `{}` invalid for type "
                    "`{}` — this column is produced by a rule, "
                    "so you may omit its distribution entirely"
                    .format(tag, kind, col.ctype))
            elif kind not in DIST_KINDS:
                if kind in RULE_KINDS:
                    problems.append(
                        "{}: `{}` is a RULE kind, not a "
                        "distribution — give this column a "
                        "normal distribution for its type and "
                        "add a top-level rule referencing it"
                        .format(tag, kind))
                else:
                    problems.append(
                        "{}: unknown distribution `{}` — valid "
                        "kinds: {}".format(
                            tag, kind, ", ".join(DIST_KINDS)))
            elif kind not in _TYPE_DISTS[col.ctype]:
                problems.append(
                    "{}: distribution `{}` invalid for type `{}`"
                    .format(tag, kind, col.ctype))
            else:
                problems.extend(self._check_params(tag, col))
            m = col.mess
            for rname in ("missing_rate", "typo_rate",
                          "format_rate", "outlier_rate",
                          "case_rate", "space_rate"):
                if not (0.0 <= getattr(m, rname) <= 1.0):
                    problems.append("{}: {} must be in [0, 1]"
                                    .format(tag, rname))
            if m.outlier_rate > 0 and col.ctype not in (
                    "int", "float"):
                problems.append("{}: outlier_rate needs a numeric "
                                "column".format(tag))
            if m.wrong_rate > 0 and col.ctype == "str_id":
                problems.append("{}: wrong_rate is not supported "
                                "on str_id columns".format(tag))
        ctypes = {c.name: c.ctype for c in self.columns}
        names = set(ctypes)
        for i, rule in enumerate(self.rules):
            rtag = "rule #{}".format(i + 1)
            kind = rule.get("kind")
            if kind not in RULE_KINDS:
                problems.append(
                    "{}: unknown kind `{}` — valid rule kinds are "
                    "date_after and derived; row duplication is "
                    "the top-level `duplicate_rate` field, not a "
                    "rule; per-column mess lives in each column's "
                    "`mess` object".format(rtag, kind))
                continue
            if kind == "date_after":
                for key in ("earlier", "later"):
                    col = rule.get(key)
                    if col not in names:
                        problems.append(
                            "{}: `{}` must name a column".format(
                                rtag, key))
                    elif ctypes[col] != "date":
                        problems.append(
                            "{}: `{}` column `{}` must have type "
                            "date (it is {})".format(
                                rtag, key, col, ctypes[col]))
                if rule.get("earlier") == rule.get("later"):
                    problems.append("{}: earlier and later must "
                                    "differ".format(rtag))
                days_from = rule.get("days_from")
                if days_from is not None:
                    if days_from not in names:
                        problems.append(
                            "{}: `days_from` must name a column"
                            .format(rtag))
                    elif ctypes[days_from] not in ("int",
                                                   "float"):
                        problems.append(
                            "{}: `days_from` column `{}` must be "
                            "numeric".format(rtag, days_from))
                else:
                    for key in ("min_days", "max_days"):
                        if key not in rule:
                            problems.append(
                                "{}: requires `{}` (or "
                                "`days_from` naming a numeric "
                                "column to drive the gap)".format(
                                    rtag, key))
            if kind == "derived":
                # A validated spec MUST plan: derived multiplies,
                # so both ends must be numeric — a live compile
                # authored derived over date columns, passed the
                # old checks, and crashed at plan time. The message
                # teaches the repair round which rule to use
                # instead.
                for key in ("target", "source"):
                    col = rule.get(key)
                    if col not in names:
                        problems.append(
                            "{}: `{}` must name a column".format(
                                rtag, key))
                    elif ctypes[col] not in ("int", "float"):
                        hint = ("for date ordering use kind "
                                "`date_after` — with `days_from` "
                                "naming a numeric column if the "
                                "gap should track that column"
                                if ctypes[col] == "date" else
                                "a generated label belongs in "
                                "the top-level `outcomes` array "
                                "(kind logistic), not in rules"
                                if ctypes[col] == "bool" else
                                "use a `column=value` indicator "
                                "coefficient in `outcomes` for "
                                "category influence")
                        problems.append(
                            "{}: `derived` requires numeric "
                            "columns, but `{}` column `{}` has "
                            "type {} — {}".format(
                                rtag, key, col, ctypes[col],
                                hint))
                if "factor" not in rule:
                    problems.append("{}: requires `factor`"
                                    .format(rtag))
                sigma = rule.get("noise_sigma", 0.0)
                if not (0.0 <= float(sigma) <= 3.0):
                    problems.append(
                        "{}: noise_sigma={} is not a noise "
                        "level — it is the sigma of "
                        "MULTIPLICATIVE lognormal noise, where "
                        "0.15 means roughly ±15% scatter and "
                        "anything above ~1 is extreme. Scale "
                        "belongs in `factor`; keep noise_sigma "
                        "in [0, 3]. (A live compile put a "
                        "dollar amount here and produced 10^150-"
                        "dollar hospital stays.)".format(
                            rtag, sigma))
                if rule.get("target") == rule.get("source"):
                    problems.append("{}: target and source must "
                                    "differ".format(rtag))
        for i, oc in enumerate(self.outcomes):
            otag = "outcome #{}".format(i + 1)
            name = oc.get("name", "")
            if not str(name).strip():
                problems.append("{}: requires a name".format(otag))
            elif name in names:
                problems.append("{}: name `{}` collides with a "
                                "column".format(otag, name))
            kind = oc.get("kind")
            if kind == "linear":
                sigma = oc.get("noise_sigma")
                if sigma is None:
                    problems.append(
                        "{}: linear outcomes require "
                        "`noise_sigma` — the additive gaussian "
                        "noise IS the known irreducible error "
                        "that sets the R^2 ceiling".format(otag))
                elif float(sigma) < 0:
                    problems.append(
                        "{}: noise_sigma must be >= 0"
                        .format(otag))
                tr = oc.get("target_range")
                if tr is not None and (
                        not isinstance(tr, (list, tuple))
                        or len(tr) != 2
                        or float(tr[0]) > float(tr[1])):
                    problems.append(
                        "{}: target_range must be [lo, hi]"
                        .format(otag))
            elif kind != "logistic":
                problems.append("{}: kind must be `logistic` "
                                "or `linear`".format(otag))
            if "intercept" not in oc:
                problems.append("{}: requires `intercept`"
                                .format(otag))
            target = oc.get("target_prevalence")
            if target is not None:
                bad = (not isinstance(target, (list, tuple))
                       or len(target) != 2)
                if not bad:
                    lo, hi = target
                    bad = not (0.0 < float(lo) <= float(hi)
                               < 1.0)
                if bad:
                    problems.append(
                        "{}: target_prevalence must be "
                        "[lo, hi] with 0 < lo <= hi < 1"
                        .format(otag))
            coeffs = oc.get("coefficients") or {}
            if not coeffs:
                problems.append("{}: requires non-empty "
                                "`coefficients`".format(otag))
            for key in coeffs:
                base = key.split("=", 1)[0]
                if base not in names:
                    problems.append(
                        "{}: coefficient `{}` names no column"
                        .format(otag, key))
                    continue
                ctype = ctypes[base]
                if "=" in key and ctype != "category":
                    problems.append(
                        "{}: indicator `{}` needs a category "
                        "column".format(otag, key))
                if "=" not in key and ctype not in (
                        "int", "float", "bool"):
                    problems.append(
                        "{}: coefficient `{}` needs a numeric or "
                        "bool column (use `{}=value` for "
                        "categories)".format(otag, key, base))
        if problems:
            raise TableSpecError("\n".join(problems))

    @staticmethod
    def _check_params(tag: str, col: ColumnSpec) -> List[str]:
        p = col.distribution
        kind = col.dist_kind()
        out: List[str] = []

        def need(*keys):
            for k in keys:
                if k not in p:
                    out.append("{}: `{}` requires param `{}`"
                               .format(tag, kind, k))
        if kind == "uniform":
            need("min", "max")
        elif kind == "normal":
            need("mean", "std")
        elif kind == "lognormal":
            need("mu", "sigma")
        elif kind == "beta":
            need("alpha", "beta")
        elif kind == "categorical":
            need("choices")
            choices = p.get("choices") or []
            weights = p.get("weights")
            if weights is not None:
                if len(weights) != len(choices):
                    out.append("{}: weights length must match "
                               "choices".format(tag))
                elif any(w < 0 for w in weights) or \
                        sum(weights) <= 0:
                    out.append("{}: weights must be non-negative "
                               "with a positive sum".format(tag))
        elif kind == "date_range":
            need("start", "end")
        elif kind == "sequence":
            need("prefix", "start")
        elif kind == "bernoulli":
            need("p")
            if "p" in p and not (0.0 <= p["p"] <= 1.0):
                out.append("{}: p must be in [0, 1]".format(tag))
        elif kind == "mixture":
            comps = p.get("components") or []
            weights = p.get("weights") or []
            if not comps:
                out.append("{}: mixture requires components"
                           .format(tag))
            if len(weights) != len(comps):
                out.append("{}: mixture weights must match "
                           "components".format(tag))
            for j, comp in enumerate(comps):
                ck = comp.get("kind")
                if ck not in DIST_KINDS or ck == "mixture":
                    out.append("{}: component #{} has invalid "
                               "kind `{}`".format(tag, j + 1, ck))
                else:
                    sub = ColumnSpec(name=col.name,
                                     ctype=col.ctype,
                                     distribution=comp)
                    out.extend(TableSpec._check_params(
                        "{} component #{}".format(tag, j + 1),
                        sub))
        return out

    # ---------------- JSON ----------------
    def to_json(self) -> str:
        return json.dumps(asdict(self), indent=2,
                          ensure_ascii=False)

    @classmethod
    def from_json(cls, raw: str) -> "TableSpec":
        d = json.loads(raw)
        return cls(
            title=d["title"],
            rows=d.get("rows", 100),
            master_seed=d.get("master_seed", 42),
            duplicate_rate=d.get("duplicate_rate", 0.0),
            rules=d.get("rules", []),
            outcomes=d.get("outcomes", []),
            columns=[
                ColumnSpec(
                    name=c["name"],
                    ctype=c["ctype"],
                    distribution=c.get("distribution", {}),
                    mess=ColumnMess(**c.get("mess", {})),
                )
                for c in d.get("columns", [])
            ],
        )


## Library: Table planning — clean truth, deterministic mess, the ledger

*Source of truth: `synthkit/tableplan.py` — this cell is generated, not hand-edited.*


In [ ]:
"""SYNTH_A1: table planning — clean truth first, mess second,
ledger always.

plan_table(spec) generates the CLEAN table (per-cell seeds keyed by
(master_seed, row_index, column_NAME) — adding a column never
changes any other column's values, growing rows never reshuffles
existing ones), then applies each column's mess policy with
independent per-cell-per-op seeds, recording every corruption:

    CellMess(row, column, op, clean, dirty)

The dirty table is what a cleaner sees; the clean table plus the
ledger is the exact answer key. Duplicate rows are appended and
ledgered with the source row index.

write_table/load_table: manifest with the verbatim spec and sha256
of every artifact (dirty.csv, clean.csv, ledger.json) — tampering
refuses to load, same law as the document corpus.

Python 3.8 compatible. Stdlib only.
"""
from __future__ import annotations

import csv
import hashlib
import io
import json
import math
import random
from dataclasses import asdict, dataclass, field
from datetime import date, timedelta
from pathlib import Path
from typing import Any, Dict, List, Tuple

pass  # intra-package import inlined above

_FIRST = ["Alex", "Sam", "Jordan", "Morgan", "Riley", "Casey",
          "Devon", "Harper", "Rowan", "Quinn", "Avery", "Jules"]
_LAST = ["Reyes", "Kim", "Okafor", "Marsh", "Ito", "Alvarez",
         "Novak", "Singh", "Bauer", "Fontaine", "Walsh", "Osei"]

_DATE_FORMATS = ["%m/%d/%Y", "%d-%b-%Y", "%B %d, %Y", "%Y.%m.%d"]


def _cell_rng(master: int, row: int, column: str,
              op: str = "value") -> random.Random:
    key = "{}:{}:{}:{}".format(master, row, column, op)
    digest = hashlib.sha256(key.encode("utf-8")).hexdigest()
    return random.Random(int(digest[:16], 16))


# ===================================================================
# Clean generation
# ===================================================================

def _gen_clean(col: ColumnSpec, row: int, master: int) -> Any:
    rng = _cell_rng(master, row, col.name)
    p = col.distribution
    kind = col.dist_kind()
    if col.ctype == "person_name":
        return "{} {}".format(rng.choice(_FIRST), rng.choice(_LAST))
    if not kind:
        return None    # rule-produced; the rules pass fills it
    if kind == "sequence":
        return "{}{}".format(p["prefix"], int(p["start"]) + row)
    if kind == "uniform":
        lo, hi = float(p["min"]), float(p["max"])
        val = rng.uniform(lo, hi)
        return int(round(val)) if col.ctype == "int" else round(
            val, 4)
    if kind == "normal":
        val = rng.gauss(float(p["mean"]), float(p["std"]))
        val = min(max(val, float(p.get("min", -math.inf))),
                  float(p.get("max", math.inf)))
        return int(round(val)) if col.ctype == "int" else round(
            val, 4)
    if kind == "lognormal":
        val = rng.lognormvariate(float(p["mu"]), float(p["sigma"]))
        val = min(max(val, float(p.get("min", 0))),
                  float(p.get("max", math.inf)))
        return int(round(val)) if col.ctype == "int" else round(
            val, 4)
    if kind == "beta":
        val = rng.betavariate(float(p["alpha"]), float(p["beta"]))
        return round(val * float(p.get("scale", 1.0)), 4)
    if kind == "categorical":
        choices = list(p["choices"])
        weights = p.get("weights")
        if weights:
            return rng.choices(choices, weights=weights, k=1)[0]
        return rng.choice(choices)
    if kind == "date_range":
        start = date.fromisoformat(p["start"])
        end = date.fromisoformat(p["end"])
        span = max((end - start).days, 0)
        return (start + timedelta(days=rng.randint(0, span)))
    if kind == "bernoulli":
        return rng.random() < float(p["p"])
    if kind == "mixture":
        pick = _cell_rng(master, row, col.name, "mixture_pick")
        comps = p["components"]
        weights = p["weights"]
        comp = pick.choices(list(range(len(comps))),
                            weights=weights, k=1)[0]
        sub = ColumnSpec(name=col.name, ctype=col.ctype,
                         distribution=comps[comp])
        return _gen_clean(sub, row, master)
    raise ValueError("unhandled distribution: {}".format(kind))


def _feature(key: str, vals: Dict[str, Any]) -> float:
    if "=" in key:
        col, want = key.split("=", 1)
        return 1.0 if str(vals[col]) == want else 0.0
    v = vals[key]
    if isinstance(v, bool):
        return 1.0 if v else 0.0
    return float(v)


def _apply_rules(spec: TableSpec, row: int,
                 vals: Dict[str, Any]) -> None:
    """Cross-column rules, in declared order, overwriting the
    target's base value. Seeds key on the TARGET column name, so
    the column-independence law survives: adding an unrelated
    column changes nothing, and a rule target depends only on its
    sources plus its own seed."""
    for i, rule in enumerate(spec.rules):
        kind = rule["kind"]
        if kind == "date_after":
            earlier = vals[rule["earlier"]]
            days_from = rule.get("days_from")
            if days_from is not None:
                # The gap IS another column's value — dates and
                # durations stay consistent, the way a real table
                # would be. (A live compile wanted exactly this
                # and the schema could not say it.)
                delta = max(int(round(
                    float(vals[days_from]))), 0)
            else:
                rng = _cell_rng(spec.master_seed, row,
                                rule["later"],
                                "rule{}".format(i))
                delta = rng.randint(int(rule["min_days"]),
                                    int(rule["max_days"]))
            vals[rule["later"]] = earlier + timedelta(days=delta)
        elif kind == "derived":
            source = vals[rule["source"]]
            base = float(source) * float(rule["factor"])
            sigma = float(rule.get("noise_sigma", 0.0))
            if sigma > 0:
                rng = _cell_rng(spec.master_seed, row,
                                rule["target"],
                                "rule{}".format(i))
                base *= rng.lognormvariate(0.0, sigma)
            target_col = next(c for c in spec.columns
                              if c.name == rule["target"])
            vals[rule["target"]] = (int(round(base))
                                    if target_col.ctype == "int"
                                    else round(base, 4))


def clean_str(value: Any) -> str:
    """Canonical string form of a clean value — the form a perfect
    cleaner should output. Dates ISO, bools True/False, floats
    without trailing zeros beyond 4 places."""
    if isinstance(value, date):
        return value.isoformat()
    if isinstance(value, bool):
        return "True" if value else "False"
    if isinstance(value, float):
        return ("{:.4f}".format(value)).rstrip("0").rstrip(".")
    return str(value)


# ===================================================================
# Mess application
# ===================================================================

def _typo(text: str, rng: random.Random) -> str:
    if len(text) < 2:
        return text + text[-1:] if text else text
    i = rng.randrange(len(text) - 1)
    op = rng.choice(("swap", "drop", "double"))
    if op == "swap":
        return text[:i] + text[i + 1] + text[i] + text[i + 2:]
    if op == "drop":
        return text[:i] + text[i + 1:]
    return text[:i] + text[i] + text[i:]


def _format_drift(col: ColumnSpec, value: Any,
                  rng: random.Random) -> str:
    if isinstance(value, date):
        return value.strftime(rng.choice(_DATE_FORMATS))
    if isinstance(value, bool):
        return rng.choice(("yes", "no")) if not value else \
            rng.choice(("yes", "Y", "TRUE", "1"))
    if isinstance(value, (int, float)):
        s = clean_str(value)
        style = rng.choice(("thousands", "currency", "spaces"))
        if style == "thousands" and isinstance(value, int) \
                and abs(value) >= 1000:
            return "{:,}".format(value)
        if style == "currency":
            return "$" + s
        return s.replace(".", " . ") if "." in s else s + " "
    return clean_str(value)


def _case_drift(text: str, rng: random.Random) -> str:
    return rng.choice((text.upper(), text.lower(), text.title()))


def _wrong_value(col: ColumnSpec, value: Any,
                 rng: random.Random) -> str:
    """A FORMAT-VALID lie: stays parseable in the column's normal
    form so no normalizer can fix it — only detection helps."""
    if isinstance(value, date):
        shift = rng.choice((-1, 1)) * rng.randint(7, 90)
        return (value + timedelta(days=shift)).isoformat()
    if isinstance(value, bool):
        return clean_str(not value)
    if isinstance(value, int):
        s = str(abs(value))
        if len(s) >= 2:
            i = rng.randrange(len(s) - 1)
            s = s[:i] + s[i + 1] + s[i] + s[i + 2:]
            out = int(s) * (1 if value >= 0 else -1)
            if out != value:
                return str(out)
        return str(value + rng.choice((-1, 1))
                   * max(1, abs(value) // 3))
    if isinstance(value, float):
        return clean_str(round(
            value * rng.uniform(1.25, 2.5)
            * rng.choice((1, 1, -1 if value < 0 else 1)), 4))
    if col.dist_kind() == "categorical":
        choices = [c for c in col.distribution["choices"]
                   if c != value]
        if choices:
            return str(rng.choice(choices))
    if col.ctype == "person_name":
        for _ in range(4):
            cand = "{} {}".format(rng.choice(_FIRST),
                                  rng.choice(_LAST))
            if cand != value:
                return cand
    return clean_str(value)


@dataclass
class CellMess:
    row: int
    column: str
    op: str          # missing|typo|format|outlier|case|space|duplicate
    clean: str
    dirty: str


@dataclass
class TableBlueprint:
    spec_title: str
    master_seed: int
    columns: List[str]
    clean_rows: List[Dict[str, str]]
    dirty_rows: List[Dict[str, str]]
    ledger: List[CellMess]
    duplicate_of: Dict[int, int] = field(default_factory=dict)
    # outcome name -> per-ORIGINAL-row true probabilities (the
    # generating model's P(y=1|x); ceiling metrics live on these)
    true_probs: Dict[str, List[float]] = field(
        default_factory=dict)
    # dirty row index -> source clean row index (for appended dups)

    def ledger_index(self) -> Dict[Tuple[int, str], CellMess]:
        return {(m.row, m.column): m for m in self.ledger
                if m.op != "duplicate"}


def plan_table(spec: TableSpec) -> TableBlueprint:
    spec.validate()
    columns = [c.name for c in spec.columns]
    clean_rows: List[Dict[str, str]] = []
    clean_vals: List[Dict[str, Any]] = []
    for r in range(spec.rows):
        vals = {c.name: _gen_clean(c, r, spec.master_seed)
                for c in spec.columns}
        _apply_rules(spec, r, vals)
        clean_vals.append(vals)
        clean_rows.append({k: clean_str(v)
                           for k, v in vals.items()})

    # Outcomes: labels from CLEAN values (truth reflects reality;
    # mess on features is what makes prediction hard).
    true_probs: Dict[str, List[float]] = {}
    for oc in spec.outcomes:
        name = oc["name"]
        kind = oc.get("kind", "logistic")
        probs: List[float] = []
        for r in range(spec.rows):
            z = float(oc["intercept"])
            for key, w in oc["coefficients"].items():
                z += float(w) * _feature(key, clean_vals[r])
            rng = _cell_rng(spec.master_seed, r, name,
                            "outcome")
            if kind == "linear":
                # The noiseless signal is the TRUTH; additive
                # gaussian noise of known sigma is the
                # irreducible error that sets the R^2 ceiling.
                probs.append(z)
                value = z + rng.gauss(0.0, float(
                    oc.get("noise_sigma", 0.0)))
                clean_vals[r][name] = round(value, 4)
                clean_rows[r][name] = clean_str(
                    round(value, 4))
            else:
                prob = 1.0 / (1.0 + math.exp(-z))
                probs.append(prob)
                label = rng.random() < prob
                clean_vals[r][name] = label
                clean_rows[r][name] = clean_str(label)
        true_probs[name] = probs
    outcome_names = [oc["name"] for oc in spec.outcomes]
    columns = columns + outcome_names

    dirty_rows: List[Dict[str, str]] = []
    ledger: List[CellMess] = []
    for r in range(spec.rows):
        row_out: Dict[str, str] = {
            name: clean_rows[r][name] for name in outcome_names}
        for col in spec.columns:
            cval = clean_vals[r][col.name]
            cstr = clean_rows[r][col.name]
            dirty = cstr
            op_applied = None
            m = col.mess
            # Ops are mutually exclusive per cell, priority order;
            # each op draws its own seeded RNG so toggling one rate
            # never shifts another's draws. `wrong` is first: a
            # plausible lie is the deepest corruption and must not
            # be masked by surface mess.
            if m.wrong_rate > 0 and _cell_rng(
                    spec.master_seed, r, col.name,
                    "wrong").random() < m.wrong_rate:
                cand = _wrong_value(col, cval, _cell_rng(
                    spec.master_seed, r, col.name, "wrong_pick"))
                if cand != cstr:
                    dirty = cand
                    op_applied = "wrong"
            elif m.missing_rate > 0 and _cell_rng(
                    spec.master_seed, r, col.name,
                    "missing").random() < m.missing_rate:
                rng = _cell_rng(spec.master_seed, r, col.name,
                                "missing_tok")
                dirty = rng.choice(m.missing_tokens)
                op_applied = "missing"
            elif m.outlier_rate > 0 and _cell_rng(
                    spec.master_seed, r, col.name,
                    "outlier").random() < m.outlier_rate:
                scaled = (cval * m.outlier_factor
                          if isinstance(cval, (int, float))
                          and not isinstance(cval, bool) else cval)
                dirty = clean_str(
                    int(scaled) if isinstance(cval, int)
                    and not isinstance(cval, bool)
                    else round(float(scaled), 4))
                op_applied = "outlier"
            elif m.format_rate > 0 and _cell_rng(
                    spec.master_seed, r, col.name,
                    "format").random() < m.format_rate:
                dirty = _format_drift(
                    col, cval, _cell_rng(spec.master_seed, r,
                                         col.name, "format_pick"))
                op_applied = "format"
            elif m.typo_rate > 0 and _cell_rng(
                    spec.master_seed, r, col.name,
                    "typo").random() < m.typo_rate:
                dirty = _typo(cstr, _cell_rng(
                    spec.master_seed, r, col.name, "typo_pick"))
                op_applied = "typo"
            elif m.case_rate > 0 and _cell_rng(
                    spec.master_seed, r, col.name,
                    "case").random() < m.case_rate:
                dirty = _case_drift(cstr, _cell_rng(
                    spec.master_seed, r, col.name, "case_pick"))
                op_applied = "case"
            elif m.space_rate > 0 and _cell_rng(
                    spec.master_seed, r, col.name,
                    "space").random() < m.space_rate:
                rng = _cell_rng(spec.master_seed, r, col.name,
                                "space_pick")
                dirty = " " * rng.randint(1, 3) + cstr + \
                    " " * rng.randint(0, 2)
                op_applied = "space"
            if op_applied and dirty != cstr or op_applied == \
                    "missing":
                ledger.append(CellMess(r, col.name, op_applied,
                                       cstr, dirty))
                row_out[col.name] = dirty
            else:
                row_out[col.name] = cstr
        dirty_rows.append(row_out)

    duplicate_of: Dict[int, int] = {}
    if spec.duplicate_rate > 0:
        n_dups = int(round(spec.rows * spec.duplicate_rate))
        rng = _cell_rng(spec.master_seed, -1, "__dups__")
        for k in range(n_dups):
            src = rng.randrange(spec.rows)
            idx = len(dirty_rows)
            dirty_rows.append(dict(dirty_rows[src]))
            duplicate_of[idx] = src
            ledger.append(CellMess(idx, "*", "duplicate",
                                   str(src), str(idx)))

    return TableBlueprint(
        spec_title=spec.title,
        master_seed=spec.master_seed,
        columns=columns,
        clean_rows=clean_rows,
        dirty_rows=dirty_rows,
        ledger=ledger,
        duplicate_of=duplicate_of,
        true_probs=true_probs,
    )


# ===================================================================
# I/O — the auditable table artifact
# ===================================================================

class TableIntegrityError(RuntimeError):
    pass


def _csv_text(columns: List[str],
              rows: List[Dict[str, str]]) -> str:
    buf = io.StringIO()
    w = csv.DictWriter(buf, fieldnames=columns,
                       lineterminator="\n")
    w.writeheader()
    for row in rows:
        w.writerow(row)
    return buf.getvalue()


def _sha(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def write_table(run_dir: Path, spec: TableSpec,
                bp: TableBlueprint) -> Path:
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    artifacts = {
        "dirty.csv": _csv_text(bp.columns, bp.dirty_rows),
        "clean.csv": _csv_text(bp.columns, bp.clean_rows),
        "ledger.json": json.dumps(
            {"ledger": [asdict(m) for m in bp.ledger],
             "duplicate_of": {str(k): v for k, v
                              in bp.duplicate_of.items()},
             "true_probs": {k: [round(x, 6) for x in v]
                            for k, v in bp.true_probs.items()}},
            indent=2),
    }
    hashes = {}
    for name, text in artifacts.items():
        (run_dir / name).write_text(text, encoding="utf-8")
        hashes[name] = _sha(text)
    manifest = {
        "synthkit_table_manifest": 1,
        "spec": json.loads(spec.to_json()),
        "rows": spec.rows,
        "dirty_rows": len(bp.dirty_rows),
        "mess_cells": len([m for m in bp.ledger
                           if m.op != "duplicate"]),
        "hashes": hashes,
    }
    (run_dir / "manifest.json").write_text(
        json.dumps(manifest, indent=2), encoding="utf-8")
    return run_dir


def load_table(run_dir: Path
               ) -> Tuple[TableSpec, TableBlueprint]:
    run_dir = Path(run_dir)
    manifest = json.loads(
        (run_dir / "manifest.json").read_text(encoding="utf-8"))
    bad = []
    texts = {}
    for name, expected in sorted(manifest["hashes"].items()):
        text = (run_dir / name).read_text(encoding="utf-8")
        if _sha(text) != expected:
            bad.append(name)
        texts[name] = text
    if bad:
        raise TableIntegrityError(
            "table integrity failed for: {}".format(
                ", ".join(bad)))
    spec = TableSpec.from_json(json.dumps(manifest["spec"]))

    def rows_of(text: str) -> List[Dict[str, str]]:
        return list(csv.DictReader(io.StringIO(text)))

    raw = json.loads(texts["ledger.json"])
    bp = TableBlueprint(
        spec_title=spec.title,
        master_seed=spec.master_seed,
        columns=[c.name for c in spec.columns],
        clean_rows=rows_of(texts["clean.csv"]),
        dirty_rows=rows_of(texts["dirty.csv"]),
        ledger=[CellMess(**m) for m in raw["ledger"]],
        duplicate_of={int(k): v for k, v
                      in raw["duplicate_of"].items()},
        true_probs=raw.get("true_probs", {}),
    )
    bp.columns = (bp.columns
                  + [oc["name"] for oc in spec.outcomes])
    return spec, bp


## Library: Cleaning evaluation — exact, cell-level, sliced by mess type

*Source of truth: `synthkit/tableeval.py` — this cell is generated, not hand-edited.*


In [ ]:
"""SYNTH_A1: cleaning evaluation — exact, cell-level, sliced.

Contract: the cleaner receives the DIRTY rows and returns the same
number of rows in the same order (duplicate handling is flag-based:
a cleaner may add a "_duplicate" column with truthy values on rows
it identifies as duplicates). Every cell is then judged against the
clean truth:

    messy cell   -> FIXED (matches clean) or MISSED
    clean cell   -> PRESERVED or OVERCORRECTED (the silent killer:
                    a "cleaner" that mangles good data)
    dup rows     -> flagged or missed (via _duplicate column)

Slices: fix_rate by mess op (missing/typo/format/outlier/case/
space) and per column — the report that says WHICH kinds of mess a
vendor's cleaner actually handles. Metric paths resolve for the
harness: overall.fix_rate, overall.overcorrection_rate,
ops.typo.fix_rate, columns.age.cell_accuracy,
duplicates.flag_rate.

Python 3.8 compatible. Stdlib only.
"""
from __future__ import annotations

import json
from dataclasses import dataclass, field
from typing import Dict, List, Optional

pass  # intra-package import inlined above
pass  # intra-package import inlined above


def _norm(s: str) -> str:
    return str(s).strip()


@dataclass
class OpScore:
    op: str
    total: int = 0
    fixed: int = 0

    @property
    def fix_rate(self) -> float:
        return self.fixed / self.total if self.total else 0.0


@dataclass
class ColumnScore:
    column: str
    cells: int = 0
    correct: int = 0

    @property
    def cell_accuracy(self) -> float:
        return self.correct / self.cells if self.cells else 0.0


@dataclass
class CleaningReport:
    cleaner_name: str
    rows: int
    mess_cells: int
    fixed: int
    missed: int
    clean_cells: int
    overcorrected: int
    ops: Dict[str, OpScore]
    columns: Dict[str, ColumnScore]
    dup_total: int
    dup_flagged: int
    dup_false_flags: int
    wrong_total: int = 0
    wrong_detected: int = 0
    suspect_false: int = 0

    @property
    def fix_rate(self) -> float:
        return self.fixed / self.mess_cells if self.mess_cells \
            else 0.0

    @property
    def overcorrection_rate(self) -> float:
        return self.overcorrected / self.clean_cells \
            if self.clean_cells else 0.0

    @property
    def wrong_detect_rate(self) -> float:
        return self.wrong_detected / self.wrong_total \
            if self.wrong_total else 0.0

    @property
    def dup_flag_rate(self) -> float:
        return self.dup_flagged / self.dup_total if self.dup_total \
            else 0.0

    def format_text(self) -> str:
        lines = [
            "CLEANING EVALUATION: {} on {} row(s), {} messy "
            "cell(s)".format(self.cleaner_name, self.rows,
                             self.mess_cells),
            "  fix rate: {:.1%}   ({} fixed / {} missed)".format(
                self.fix_rate, self.fixed, self.missed),
            "  overcorrection: {:.2%} of {} clean cell(s) "
            "damaged".format(self.overcorrection_rate,
                             self.clean_cells),
        ]
        if self.wrong_total:
            lines.append(
                "  wrong-value detection: {:.1%} ({}/{}, {} false "
                "suspicion(s))".format(
                    self.wrong_detect_rate, self.wrong_detected,
                    self.wrong_total, self.suspect_false))
        if self.dup_total:
            lines.append(
                "  duplicates flagged: {:.1%} ({}/{}, {} false "
                "flag(s))".format(self.dup_flag_rate,
                                  self.dup_flagged, self.dup_total,
                                  self.dup_false_flags))
        lines.append("")
        lines.append("FIX RATE BY MESS TYPE:")
        for op, sc in sorted(self.ops.items()):
            lines.append("  {:<10} {:>6.1%}  ({}/{})".format(
                op, sc.fix_rate, sc.fixed, sc.total))
        lines.append("")
        lines.append("CELL ACCURACY BY COLUMN:")
        for name, sc in sorted(self.columns.items()):
            lines.append("  {:<20} {:>6.1%}".format(
                name, sc.cell_accuracy))
        return "\n".join(lines)

    def to_json(self) -> str:
        return json.dumps({
            "cleaner": self.cleaner_name,
            "fix_rate": round(self.fix_rate, 4),
            "overcorrection_rate": round(
                self.overcorrection_rate, 4),
            "dup_flag_rate": round(self.dup_flag_rate, 4),
            "ops": {k: {"fix_rate": round(v.fix_rate, 4),
                        "total": v.total}
                    for k, v in self.ops.items()},
            "columns": {k: round(v.cell_accuracy, 4)
                        for k, v in self.columns.items()},
        }, indent=2)


def evaluate_cleaning(bp: TableBlueprint,
                      cleaned_rows: List[Dict[str, str]],
                      cleaner_name: str = "cleaner",
                      ) -> CleaningReport:
    if len(cleaned_rows) != len(bp.dirty_rows):
        raise ValueError(
            "cleaner must return {} row(s) in order, got {} — "
            "flag duplicates via a `_duplicate` column instead of "
            "dropping rows".format(len(bp.dirty_rows),
                                   len(cleaned_rows)))
    mess = bp.ledger_index()
    n_originals = len(bp.clean_rows)
    ops: Dict[str, OpScore] = {}
    cols: Dict[str, ColumnScore] = {
        c: ColumnScore(c) for c in bp.columns}
    fixed = missed = clean_cells = overcorrected = 0

    for r in range(n_originals):
        for c in bp.columns:
            truth = _norm(bp.clean_rows[r][c])
            out = _norm(cleaned_rows[r].get(c, ""))
            entry = mess.get((r, c))
            col_sc = cols[c]
            col_sc.cells += 1
            if entry is not None:
                sc = ops.setdefault(entry.op, OpScore(entry.op))
                sc.total += 1
                if out == truth:
                    sc.fixed += 1
                    fixed += 1
                    col_sc.correct += 1
                else:
                    missed += 1
            else:
                clean_cells += 1
                if out == truth:
                    col_sc.correct += 1
                else:
                    overcorrected += 1

    dup_total = len(bp.duplicate_of)
    dup_flagged = dup_false = 0
    # Wrong-value detection: the cleaner may emit a `_suspect`
    # column per row listing comma-separated column names it
    # believes carry wrong values. Fixing a plausible lie is
    # usually impossible; FLAGGING it is the measurable skill.
    wrong_cells = {(m.row, m.column) for m in bp.ledger
                   if m.op == "wrong"}
    wrong_detected = suspect_false = 0
    for idx, row in enumerate(cleaned_rows):
        flagged = str(row.get("_duplicate", "")).strip().lower() \
            in ("1", "true", "yes", "y")
        if idx in bp.duplicate_of:
            if flagged:
                dup_flagged += 1
        elif flagged:
            dup_false += 1
        suspects = [s.strip() for s in
                    str(row.get("_suspect", "")).split(",")
                    if s.strip()]
        for col in suspects:
            if (idx, col) in wrong_cells:
                wrong_detected += 1
            elif idx < n_originals:
                suspect_false += 1

    return CleaningReport(
        cleaner_name=cleaner_name,
        rows=len(bp.dirty_rows),
        mess_cells=len(mess),
        fixed=fixed,
        missed=missed,
        clean_cells=clean_cells,
        overcorrected=overcorrected,
        ops=ops,
        columns=cols,
        dup_total=dup_total,
        dup_flagged=dup_flagged,
        dup_false_flags=dup_false,
        wrong_total=len(wrong_cells),
        wrong_detected=wrong_detected,
        suspect_false=suspect_false,
    )


# ===================================================================
# Harness integration
# ===================================================================

def resolve_table_metric(report: CleaningReport,
                         path: str) -> float:
    parts = path.split(".")
    try:
        if parts[0] == "overall":
            return {"fix_rate": report.fix_rate,
                    "overcorrection_rate":
                        report.overcorrection_rate}[parts[1]]
        if parts[0] == "ops":
            return {"fix_rate": report.ops[parts[1]].fix_rate,
                    "total": float(report.ops[parts[1]].total)
                    }[parts[2]]
        if parts[0] == "columns":
            return report.columns[parts[1]].cell_accuracy
        if parts[0] == "wrong":
            return {"detect_rate": report.wrong_detect_rate,
                    "false_suspects":
                        float(report.suspect_false),
                    "total": float(report.wrong_total)}[parts[1]]
        if parts[0] == "duplicates":
            return {"flag_rate": report.dup_flag_rate,
                    "false_flags":
                        float(report.dup_false_flags)}[parts[1]]
    except KeyError:
        raise MetricError(
            "metric not present in this report: {}".format(path))
    raise MetricError("unknown metric family: {}".format(path))


def resolve_table_counts(report: CleaningReport, path: str):
    """(k, n) behind a proportion metric, for interval-aware
    verdicts. None for non-proportion paths."""
    parts = path.split(".")
    if parts[0] == "overall":
        if parts[1] == "fix_rate":
            return (report.fixed, report.mess_cells)
        if parts[1] == "overcorrection_rate":
            return (report.overcorrected, report.clean_cells)
    if parts[0] == "ops" and parts[2] == "fix_rate" \
            and parts[1] in report.ops:
        sc = report.ops[parts[1]]
        return (sc.fixed, sc.total)
    if parts[0] == "wrong" and parts[1] == "detect_rate":
        return (report.wrong_detected, report.wrong_total)
    if parts[0] == "duplicates" and parts[1] == "flag_rate":
        return (report.dup_flagged, report.dup_total)
    return None


@dataclass
class RegressionReport:
    """Continuous-outcome scoring against the planted signal.
    ceiling_r2 = Var(signal) / Var(realized y): the fraction of
    outcome variance that IS signal — no model can honestly
    exceed it, because the rest is noise by construction."""
    solver_name: str
    outcome: str
    n: int
    r2: float
    rmse: float
    mae: float
    ceiling_r2: float

    @property
    def r2_gap(self) -> float:
        return round(self.ceiling_r2 - self.r2, 4)

    def format_text(self) -> str:
        return ("REGRESSION EVALUATION: {} on `{}` ({} row(s))\n"
                "  R^2:   {:.3f}   (ceiling {:.3f} — gap {:.3f})"
                "\n  RMSE:  {:.4f}   MAE: {:.4f}".format(
                    self.solver_name, self.outcome, self.n,
                    self.r2, self.ceiling_r2, self.r2_gap,
                    self.rmse, self.mae))


def evaluate_regression(bp, outcome: str,
                        predictions: List[float],
                        solver_name: str) -> RegressionReport:
    if outcome not in bp.true_probs:
        raise TableEvalError(
            "`{}` is not a generated outcome of this table"
            .format(outcome))
    n = len(bp.clean_rows)
    if len(predictions) != n:
        raise TableEvalError(
            "predictions must cover every clean row: got {} "
            "for {} rows".format(len(predictions), n))
    y = [float(r[outcome]) for r in bp.clean_rows]
    signal = bp.true_probs[outcome]
    mean_y = sum(y) / n
    ss_tot = sum((v - mean_y) ** 2 for v in y) or 1e-12
    ss_res = sum((yv - float(pv)) ** 2
                 for yv, pv in zip(y, predictions))
    mean_sig = sum(signal) / n
    ss_sig = sum((s - mean_sig) ** 2 for s in signal)
    rmse = (ss_res / n) ** 0.5
    mae = sum(abs(yv - float(pv))
              for yv, pv in zip(y, predictions)) / n
    return RegressionReport(
        solver_name=solver_name, outcome=outcome, n=n,
        r2=round(1.0 - ss_res / ss_tot, 4),
        rmse=round(rmse, 4), mae=round(mae, 4),
        ceiling_r2=round(min(ss_sig / ss_tot, 1.0), 4))


def resolve_regression_metric(report: RegressionReport,
                              path: str) -> float:
    mapping = {
        "regress.r2": report.r2,
        "regress.rmse": report.rmse,
        "regress.mae": report.mae,
        "regress.r2_gap": report.r2_gap,
        "regress.ceiling_r2": report.ceiling_r2,
    }
    if path not in mapping:
        raise TableEvalError(
            "unknown regression metric `{}` — known: {}".format(
                path, ", ".join(sorted(mapping))))
    return mapping[path]


def resolve_prediction_counts(report: "PredictionReport",
                              path: str):
    """AUROC metrics resolve to ("auroc", value, n_pos, n_neg);
    others None."""
    if path in ("predict.auroc",):
        return ("auroc", report.auroc, report.n_pos,
                report.n - report.n_pos)
    return None


@dataclass
class TableExperiment:
    name: str
    spec: "TableSpec"                     # noqa: F821
    cleaner: object                       # callable(rows)->rows
    cleaner_name: str
    conditions: List[Condition] = field(default_factory=list)


@dataclass
class TableExperimentResult:
    name: str
    passed: bool
    measured: Dict[str, float]
    failed_conditions: List[str]
    report: CleaningReport
    measured_descriptions: List[str] = field(default_factory=list)

    def finding(self) -> str:
        lines = ["Table experiment '{}' on {} row(s): {}.".format(
            self.name, self.report.rows,
            "PASSED" if self.passed else "FAILED")]
        lines += ["  " + d for d in self.measured_descriptions]
        return "\n".join(lines)


def run_table_experiment(exp: TableExperiment
                         ) -> TableExperimentResult:
    pass  # intra-package import inlined above
    bp = plan_table(exp.spec)
    cleaned = exp.cleaner([dict(r) for r in bp.dirty_rows])
    report = evaluate_cleaning(bp, cleaned, exp.cleaner_name)
    measured: Dict[str, float] = {}
    descriptions: List[str] = []
    failed: List[str] = []
    for cond in exp.conditions:
        value = resolve_table_metric(report, cond.metric)
        measured[cond.metric] = round(value, 4)
        descriptions.append(cond.describe(value))
        if not cond.holds(value):
            failed.append(cond.metric)
    result = TableExperimentResult(
        name=exp.name, passed=not failed, measured=measured,
        failed_conditions=failed, report=report)
    result.measured_descriptions = descriptions
    return result


# ===================================================================
# Prediction evaluation — scored against the ceiling.
# ===================================================================

@dataclass
class PredictionReport:
    solver_name: str
    outcome: str
    n: int
    n_pos: int
    auroc: float
    ceiling_auroc: float
    brier_score: float
    accuracy: float
    f1: float

    @property
    def auroc_gap(self) -> float:
        return self.ceiling_auroc - self.auroc

    def format_text(self) -> str:
        return (
            "PREDICTION EVALUATION: {} on `{}` ({} row(s), {} "
            "positive)\n"
            "  AUROC:   {:.3f}   (ceiling {:.3f} — gap {:.3f})\n"
            "  Brier:   {:.4f}\n"
            "  acc@0.5: {:.1%}   F1: {:.3f}".format(
                self.solver_name, self.outcome, self.n,
                self.n_pos, self.auroc, self.ceiling_auroc,
                self.auroc_gap, self.brier_score, self.accuracy,
                self.f1))


def evaluate_prediction(bp: TableBlueprint, outcome: str,
                        scores: List[float],
                        solver_name: str = "solver",
                        ) -> PredictionReport:
    pass  # intra-package import inlined above
    if outcome not in bp.true_probs:
        raise ValueError(
            "outcome `{}` not generated in this table (have: {})"
            .format(outcome, sorted(bp.true_probs)))
    n = len(bp.clean_rows)
    if len(scores) != n:
        raise ValueError(
            "need one score per ORIGINAL row ({}) in order, "
            "got {}".format(n, len(scores)))
    labels = [1 if bp.clean_rows[r][outcome] == "True" else 0
              for r in range(n)]
    thresh = at_threshold(scores, labels)
    return PredictionReport(
        solver_name=solver_name,
        outcome=outcome,
        n=n,
        n_pos=sum(labels),
        auroc=auroc(scores, labels),
        ceiling_auroc=ceiling_auroc(
            bp.true_probs[outcome], labels),
        brier_score=brier(
            [min(max(s, 0.0), 1.0) for s in scores], labels),
        accuracy=thresh["accuracy"],
        f1=thresh["f1"],
    )


def resolve_prediction_metric(report: PredictionReport,
                              path: str) -> float:
    try:
        return {
            "predict.auroc": report.auroc,
            "predict.ceiling_auroc": report.ceiling_auroc,
            "predict.auroc_gap": report.auroc_gap,
            "predict.brier": report.brier_score,
            "predict.accuracy": report.accuracy,
            "predict.f1": report.f1,
        }[path]
    except KeyError:
        raise MetricError(
            "unknown prediction metric: {}".format(path))


## Library: ML metrics — stdlib AUROC, Brier, and the ceiling

*Source of truth: `synthkit/mlmetrics.py` — this cell is generated, not hand-edited.*


In [ ]:
"""SYNTH_B1: ML metrics — stdlib, exact, ceiling-aware.

AUROC via the rank formulation (Mann-Whitney) with average ranks
for ties; Brier score; accuracy/precision/recall at a threshold.

The synthkit-only capability: because outcomes are GENERATED from a
known model, every row has a true probability — so ceiling metrics
(the score of the Bayes-optimal classifier that knows the true
probabilities) are computable for any dataset synthkit makes. A
vendor's 0.71 means something entirely different against a ceiling
of 0.74 than against 0.92.

Python 3.8 compatible. Stdlib only.
"""
from __future__ import annotations

from typing import Dict, List, Sequence


class MetricInputError(ValueError):
    pass


def _check(scores: Sequence[float], labels: Sequence[int]) -> None:
    if len(scores) != len(labels):
        raise MetricInputError(
            "scores and labels must align: {} vs {}".format(
                len(scores), len(labels)))
    if not scores:
        raise MetricInputError("empty inputs")
    for y in labels:
        if y not in (0, 1, True, False):
            raise MetricInputError(
                "labels must be binary, got {!r}".format(y))


def auroc(scores: Sequence[float], labels: Sequence[int]) -> float:
    """Probability a random positive outscores a random negative
    (ties count half). Rank-based; O(n log n)."""
    _check(scores, labels)
    n_pos = sum(1 for y in labels if y)
    n_neg = len(labels) - n_pos
    if n_pos == 0 or n_neg == 0:
        raise MetricInputError(
            "AUROC needs both classes present ({} pos, {} neg)"
            .format(n_pos, n_neg))
    order = sorted(range(len(scores)), key=lambda i: scores[i])
    ranks = [0.0] * len(scores)
    i = 0
    while i < len(order):
        j = i
        while j + 1 < len(order) and \
                scores[order[j + 1]] == scores[order[i]]:
            j += 1
        avg = (i + j) / 2.0 + 1.0
        for k in range(i, j + 1):
            ranks[order[k]] = avg
        i = j + 1
    rank_sum = sum(r for r, y in zip(ranks, labels) if y)
    u = rank_sum - n_pos * (n_pos + 1) / 2.0
    return u / (n_pos * n_neg)


def brier(probs: Sequence[float], labels: Sequence[int]) -> float:
    _check(probs, labels)
    return sum((p - (1.0 if y else 0.0)) ** 2
               for p, y in zip(probs, labels)) / len(probs)


def at_threshold(scores: Sequence[float], labels: Sequence[int],
                 threshold: float = 0.5) -> Dict[str, float]:
    _check(scores, labels)
    tp = fp = tn = fn = 0
    for s, y in zip(scores, labels):
        pred = s >= threshold
        if pred and y:
            tp += 1
        elif pred:
            fp += 1
        elif y:
            fn += 1
        else:
            tn += 1
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = (2 * precision * recall / (precision + recall)
          if precision + recall else 0.0)
    return {
        "accuracy": (tp + tn) / len(scores),
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


def ceiling_auroc(true_probs: Sequence[float],
                  labels: Sequence[int]) -> float:
    """The Bayes-optimal AUROC on this realized dataset: the score
    of a classifier that outputs the TRUE generating probability
    for every row. No real model can beat it except by luck."""
    return auroc(true_probs, labels)


# ===================================================================
# Uncertainty — every rate is k-of-n, and verdicts should know it.
# A live bake-off flipped a bar-edge verdict between identical
# runs because 0.105 on n=19 (CI roughly [0.03, 0.29]) never
# contained enough information to resolve a bar of 0.10.
# ===================================================================

import math as _math


def wilson_interval(k: int, n: int, z: float = 1.96):
    """Wilson score interval for a binomial proportion. Behaves
    sanely at the edges (k=0, k=n) where the normal approximation
    lies."""
    if n <= 0:
        return (0.0, 1.0)
    if not (0 <= k <= n):
        raise MetricInputError(
            "k must be within [0, n], got k={} n={}".format(k, n))
    p = k / n
    denom = 1.0 + z * z / n
    center = (p + z * z / (2 * n)) / denom
    half = (z * _math.sqrt(
        p * (1 - p) / n + z * z / (4 * n * n))) / denom
    return (max(0.0, center - half), min(1.0, center + half))


def format_rate(k: int, n: int, z: float = 1.96) -> str:
    lo, hi = wilson_interval(k, n, z)
    p = k / n if n else 0.0
    return "{:.3f} [{:.3f}, {:.3f}] n={}".format(p, lo, hi, n)


def required_n(p_obs: float, bar: float, z: float = 1.96):
    """Approximate sample size at which a Wilson-style interval
    around p_obs would exclude the bar — the PRESCRIPTION: how
    much more data resolves an inconclusive verdict. None when
    the observed rate sits on the bar (no n resolves a tie)."""
    gap = abs(p_obs - bar)
    if gap < 1e-9:
        return None
    p = min(max(p_obs, 1e-6), 1 - 1e-6)
    n = (z * z * p * (1 - p)) / (gap * gap)
    return int(_math.ceil(n))


def auroc_interval(a: float, n_pos: int, n_neg: int,
                   z: float = 1.96):
    """Hanley-McNeil confidence interval for AUROC."""
    if n_pos <= 0 or n_neg <= 0:
        return (0.0, 1.0)
    a = min(max(a, 1e-6), 1 - 1e-6)
    q1 = a / (2 - a)
    q2 = 2 * a * a / (1 + a)
    var = (a * (1 - a) + (n_pos - 1) * (q1 - a * a)
           + (n_neg - 1) * (q2 - a * a)) / (n_pos * n_neg)
    half = z * _math.sqrt(max(var, 0.0))
    return (max(0.0, a - half), min(1.0, a + half))


## Library: Campaigns — a stated goal becomes a tier ladder

*Source of truth: `synthkit/campaign.py` — this cell is generated, not hand-edited.*


In [ ]:
"""SYNTH_B1: campaigns — a stated goal becomes a tier ladder.

The user says WHAT they want to test ("can this vendor clean?",
"can this model predict?", "can this extractor read?"); the
campaign compiler turns a base spec into escalating tiers with
per-tier bars:

    clean:   easy (half mess, no lies) -> standard (as specced)
             -> adversarial (heavier mess + wrong values that must
             be DETECTED)
    predict: strong signal -> as specced -> weak signal + messier
             features; every tier reports the ceiling AUROC, and
             prediction runs SPLIT-BLINDED: the solver trains on a
             table from a shifted seed and is scored on the base
             table it has never seen.
    extract: documents — easier difficulty mix and fewer traps ->
             as specced -> hard-shifted mix with denser distractors

run_campaign walks the ladder in order and reports the highest
tier passed, with per-tier evidence. Deterministic end to end.

Python 3.8 compatible. Stdlib only.
"""
from __future__ import annotations

import json
from dataclasses import dataclass, field
from typing import Any, Callable, Dict, List, Optional

pass  # intra-package import inlined above
pass  # intra-package import inlined above

GOALS = ("clean", "predict", "extract", "regress")


class CampaignError(ValueError):
    pass


@dataclass
class Tier:
    name: str
    spec_json: str                 # TableSpec or DataSpec JSON
    conditions: List[str]
    notes: str = ""


@dataclass
class Campaign:
    goal: str
    title: str
    tiers: List[Tier]
    outcome: str = ""              # predict goal only


@dataclass
class TierResult:
    tier: str
    passed: bool
    measured: Dict[str, float]
    failed_conditions: List[str]
    evidence: str


@dataclass
class CampaignResult:
    goal: str
    title: str
    tier_results: List[TierResult]

    @property
    def highest_passed(self) -> int:
        n = 0
        for tr in self.tier_results:
            if not tr.passed:
                break
            n += 1
        return n

    def format_text(self) -> str:
        total = len(self.tier_results)
        lines = ["CAMPAIGN [{}]: {}".format(self.goal, self.title),
                 "  cleared tier {} of {}".format(
                     self.highest_passed, total)]
        for i, tr in enumerate(self.tier_results):
            lines.append("  tier {} `{}`: {}".format(
                i + 1, tr.tier,
                "PASSED" if tr.passed else "FAILED"))
            for line in tr.evidence.splitlines():
                lines.append("    " + line)
        return "\n".join(lines)


# ===================================================================
# Compilation: base spec -> escalating tiers
# ===================================================================

def _scaled_table(spec: TableSpec, mess_factor: float,
                  wrong_on: bool, signal_factor: float = 1.0,
                  ) -> TableSpec:
    out = TableSpec.from_json(spec.to_json())

    def clamp(x):
        return min(max(x, 0.0), 0.95)
    for col in out.columns:
        m = col.mess
        m.missing_rate = clamp(m.missing_rate * mess_factor)
        m.typo_rate = clamp(m.typo_rate * mess_factor)
        m.format_rate = clamp(m.format_rate * mess_factor)
        m.outlier_rate = clamp(m.outlier_rate * mess_factor)
        m.case_rate = clamp(m.case_rate * mess_factor)
        m.space_rate = clamp(m.space_rate * mess_factor)
        m.wrong_rate = clamp(m.wrong_rate * mess_factor) \
            if wrong_on else 0.0
    out.duplicate_rate = min(
        out.duplicate_rate * mess_factor, 0.5)
    for oc in out.outcomes:
        oc["coefficients"] = {
            k: v * signal_factor
            for k, v in oc["coefficients"].items()}
    return out


def compile_campaign(goal: str, base_spec: Any,
                     bars: Optional[Dict[str, float]] = None,
                     outcome: str = "") -> Campaign:
    """bars (all optional, sensible defaults):
    clean:   fix_rate, overcorrection_max, detect_rate
    predict: auroc (per-tier bars derived), gap_max
    extract: recall, trap_max
    """
    bars = dict(bars or {})
    if goal not in GOALS:
        raise CampaignError(
            "unknown goal `{}` (valid: {})".format(
                goal, ", ".join(GOALS)))
    if goal == "clean":
        if not isinstance(base_spec, TableSpec):
            raise CampaignError("clean campaigns need a TableSpec")
        fix = bars.get("fix_rate", 0.9)
        over = bars.get("overcorrection_max", 0.01)
        detect = bars.get("detect_rate", 0.5)
        base = [
            "overall.fix_rate >= {}".format(fix),
            "overall.overcorrection_rate <= {}".format(over),
        ]
        tiers = [
            Tier("light-mess",
                 _scaled_table(base_spec, 0.5, False).to_json(),
                 list(base),
                 "half mess rates, no wrong values, no "
                 "duplicates" if base_spec.duplicate_rate == 0
                 else "half mess rates, no wrong values"),
            Tier("as-specified",
                 _scaled_table(base_spec, 1.0, False).to_json(),
                 list(base), "full mess rates, no wrong values"),
            Tier("adversarial",
                 _scaled_table(base_spec, 1.5, True).to_json(),
                 base + ["wrong.detect_rate >= {}".format(detect)],
                 "heavier mess plus format-valid wrong values "
                 "that must be detected"),
        ]
        return Campaign(goal, "cleaning: {}".format(
            base_spec.title), tiers)

    if goal == "predict":
        if not isinstance(base_spec, TableSpec):
            raise CampaignError(
                "predict campaigns need a TableSpec")
        if not outcome or not any(
                oc.get("name") == outcome
                for oc in base_spec.outcomes):
            raise CampaignError(
                "predict campaigns need `outcome` naming a "
                "generated outcome")
        bar = bars.get("auroc", 0.7)
        gap = bars.get("gap_max", 0.15)
        tiers = [
            Tier("strong-signal",
                 _scaled_table(base_spec, 0.5, False,
                               signal_factor=1.5).to_json(),
                 ["predict.auroc >= {}".format(bar),
                  "predict.auroc_gap <= {}".format(gap)],
                 "amplified coefficients, light feature mess"),
            Tier("as-specified",
                 _scaled_table(base_spec, 1.0, False,
                               signal_factor=1.0).to_json(),
                 ["predict.auroc >= {}".format(bar),
                  "predict.auroc_gap <= {}".format(gap)],
                 "signal and mess as declared"),
            Tier("weak-signal",
                 _scaled_table(base_spec, 1.25, False,
                               signal_factor=0.6).to_json(),
                 ["predict.auroc >= {}".format(
                     round(bar - 0.1, 3)),
                  "predict.auroc_gap <= {}".format(gap)],
                 "attenuated coefficients, messier features; the "
                 "bar drops but the CEILING drops more — the gap "
                 "condition is what still bites"),
        ]
        return Campaign(goal, "prediction of `{}`: {}".format(
            outcome, base_spec.title), tiers, outcome=outcome)

    if goal == "regress":
        if not isinstance(base_spec, TableSpec):
            raise CampaignError(
                "regress campaigns need a TableSpec")
        linear = [oc for oc in base_spec.outcomes
                  if oc.get("name") == outcome
                  and oc.get("kind") == "linear"]
        if not outcome or not linear:
            raise CampaignError(
                "regress campaigns need `outcome` naming a "
                "kind-linear generated outcome")
        bar = bars.get("r2", 0.5)
        gap = bars.get("gap_max", 0.2)
        conds = ["regress.r2 >= {}".format(bar),
                 "regress.r2_gap <= {}".format(gap)]
        tiers = [
            Tier("strong-signal",
                 _scaled_table(base_spec, 0.5, False,
                               signal_factor=1.5).to_json(),
                 list(conds),
                 "amplified coefficients, light feature mess"),
            Tier("as-specified",
                 _scaled_table(base_spec, 1.0, False,
                               signal_factor=1.0).to_json(),
                 list(conds),
                 "signal and mess as declared"),
            Tier("weak-signal",
                 _scaled_table(base_spec, 1.25, False,
                               signal_factor=0.6).to_json(),
                 ["regress.r2 >= {}".format(
                     round(bar - 0.15, 3)),
                  "regress.r2_gap <= {}".format(gap)],
                 "attenuated coefficients, messier features; "
                 "the R^2 ceiling drops with the signal — the "
                 "gap condition is what still bites"),
        ]
        return Campaign(goal, "regression of `{}`: {}".format(
            outcome, base_spec.title), tiers, outcome=outcome)

    # extract: documents
    pass  # intra-package import inlined above
    if not isinstance(base_spec, DataSpec):
        raise CampaignError("extract campaigns need a DataSpec")
    recall = bars.get("recall", 0.85)
    trap = bars.get("trap_max", 0.1)

    _DEMOTE = {"hard": "medium", "medium": "easy",
               "easy": "easy"}
    _PROMOTE = {"easy": "medium", "medium": "hard",
                "hard": "hard"}

    def scaled_doc(shift: str, distractor_factor: float) -> str:
        d = json.loads(base_spec.to_json())
        table = {"down": _DEMOTE, "none": {}, "up": _PROMOTE}[
            shift]
        for uf in d.get("unstructured_fields", []):
            for el in uf.get("target_elements", []):
                if table:
                    el["difficulty"] = table.get(
                        el.get("difficulty", "medium"),
                        el.get("difficulty"))
            for dis in uf.get("distractors", []):
                dis["density"] = round(min(max(
                    dis.get("density", 0.0) * distractor_factor,
                    0.0), 1.0), 3)
        return json.dumps(d)

    conds = ["overall_recall >= {}".format(recall)]
    trap_conds = []
    for uf in json.loads(base_spec.to_json()).get(
            "unstructured_fields", []):
        for dis in uf.get("distractors", []):
            trap_conds.append(
                "distractors.{}.fp_rate <= {}".format(
                    dis["distractor_id"], trap))
    tiers = [
        Tier("gentle", scaled_doc("down", 0.5), list(conds),
             "element difficulty demoted a step, distractors "
             "halved"),
        Tier("as-specified", scaled_doc("none", 1.0),
             conds + trap_conds, "the spec as written"),
        Tier("adversarial", scaled_doc("up", 1.5),
             conds + trap_conds,
             "element difficulty promoted a step, distractors "
             "amplified"),
    ]
    return Campaign(goal, "extraction: {}".format(
        base_spec.title), tiers)


# ===================================================================
# Persistence — campaigns and results as auditable artifacts
# ===================================================================

class CampaignIntegrityError(RuntimeError):
    pass


def _sha(text: str) -> str:
    import hashlib
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def write_campaign(run_dir, campaign: Campaign,
                   result: Optional[CampaignResult] = None,
                   result_name: str = ""):
    """Results are APPEND-ONLY study records: each run lands in
    results/trial_NNN_<name>.json (a live bake-off clobbered its
    control arm before this existed). result.json remains the
    latest, for compatibility."""
    from pathlib import Path as _P
    run_dir = _P(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    artifacts = {
        "campaign.json": json.dumps({
            "goal": campaign.goal,
            "title": campaign.title,
            "outcome": campaign.outcome,
            "tiers": [{"name": t.name,
                       "spec": json.loads(t.spec_json),
                       "conditions": t.conditions,
                       "notes": t.notes}
                      for t in campaign.tiers],
        }, indent=2),
    }
    if result is not None:
        payload = json.dumps({
            "solver": result_name or "unnamed",
            "highest_passed": result.highest_passed,
            "tiers": [{"tier": tr.tier, "passed": tr.passed,
                       "measured": tr.measured,
                       "failed_conditions": tr.failed_conditions,
                       "evidence": tr.evidence}
                      for tr in result.tier_results],
            "text": result.format_text(),
        }, indent=2)
        artifacts["result.json"] = payload
        results_dir = run_dir / "results"
        results_dir.mkdir(exist_ok=True)
        n = len(list(results_dir.glob("trial_*.json"))) + 1
        safe = "".join(c if c.isalnum() or c in "-_" else "-"
                       for c in (result_name or "unnamed"))[:40]
        (results_dir / "trial_{:03d}_{}.json".format(
            n, safe)).write_text(payload, encoding="utf-8")
    hashes = {}
    for name, text in artifacts.items():
        (run_dir / name).write_text(text, encoding="utf-8")
        hashes[name] = _sha(text)
    (run_dir / "manifest.json").write_text(json.dumps({
        "synthkit_campaign_manifest": 1,
        "hashes": hashes,
    }, indent=2), encoding="utf-8")
    return run_dir


def read_trials(run_dir) -> List[dict]:
    from pathlib import Path as _P
    results_dir = _P(run_dir) / "results"
    trials = []
    if results_dir.is_dir():
        for path in sorted(results_dir.glob("trial_*.json")):
            trials.append(json.loads(
                path.read_text(encoding="utf-8")))
    return trials


def format_trials(run_dir) -> str:
    trials = read_trials(run_dir)
    if not trials:
        return "no trials recorded yet"
    tier_names = [t["tier"] for t in trials[0]["tiers"]]
    lines = ["TRIALS ({} arm(s)):".format(len(trials))]
    header = "  {:<24}".format("arm") + "".join(
        "{:<20}".format(n[:18]) for n in tier_names) + "cleared"
    lines.append(header)
    for tr in trials:
        cells = []
        for tier in tr["tiers"]:
            mark = "PASS" if tier["passed"] else "FAIL"
            fails = tier["failed_conditions"]
            if fails:
                key = fails[0]
                val = tier["measured"].get(key)
                mark += " {}={}".format(
                    key.split(".")[-1], val)
            cells.append("{:<20}".format(mark[:18]))
        lines.append("  {:<24}{}{}/{}".format(
            tr["solver"][:22], "".join(cells),
            tr["highest_passed"], len(tr["tiers"])))
    return "\n".join(lines)


def load_campaign(run_dir) -> Campaign:
    from pathlib import Path as _P
    run_dir = _P(run_dir)
    manifest = json.loads(
        (run_dir / "manifest.json").read_text(encoding="utf-8"))
    bad = []
    for name, expected in sorted(manifest["hashes"].items()):
        text = (run_dir / name).read_text(encoding="utf-8")
        if _sha(text) != expected:
            bad.append(name)
    if bad:
        raise CampaignIntegrityError(
            "campaign integrity failed for: {}".format(
                ", ".join(bad)))
    d = json.loads(
        (run_dir / "campaign.json").read_text(encoding="utf-8"))
    return Campaign(
        goal=d["goal"], title=d["title"],
        outcome=d.get("outcome", ""),
        tiers=[Tier(name=t["name"],
                    spec_json=json.dumps(t["spec"]),
                    conditions=t["conditions"],
                    notes=t.get("notes", ""))
               for t in d["tiers"]],
    )


# ===================================================================
# Running
# ===================================================================

def run_campaign(campaign: Campaign,
                 solver: Callable,
                 solver_name: str = "solver",
                 train_seed_offset: int = 1000,
                 ) -> CampaignResult:
    """clean: solver(dirty_rows) -> cleaned rows.
    predict: solver(train_rows, train_labels, test_rows) ->
    scores; SPLIT-BLINDED — trains on a shifted-seed table, scored
    on the tier table. extract: solver is an extractor fn(text) ->
    list of {element, value}; the tier runs stub-rendered."""
    results: List[TierResult] = []
    for tier in campaign.tiers:
        if campaign.goal == "clean":
            tr = _run_clean_tier(campaign, tier, solver,
                                 solver_name)
        elif campaign.goal == "predict":
            tr = _run_predict_tier(campaign, tier, solver,
                                   solver_name,
                                   train_seed_offset)
        elif campaign.goal == "regress":
            tr = _run_regress_tier(campaign, tier, solver,
                                   solver_name,
                                   train_seed_offset)
        else:
            tr = _run_extract_tier(campaign, tier, solver,
                                   solver_name)
        results.append(tr)
    return CampaignResult(goal=campaign.goal,
                          title=campaign.title,
                          tier_results=results)


def _judge(conditions: List[str], resolver,
           counts_resolver=None) -> TierResult:
    """Point-estimate pass/fail gates the ladder (compat), but
    each evidence line carries the interval and a three-way
    reading: DECISIVE when the whole CI sits on one side of the
    bar, INCONCLUSIVE otherwise — with a PRESCRIPTION for how
    much data would resolve it. Synthkit can generate that data,
    so the prescription is executable."""
    pass  # intra-package import inlined above
    measured: Dict[str, float] = {}
    failed: List[str] = []
    lines: List[str] = []
    for raw in conditions:
        cond = Condition.parse(raw)
        value = resolver(cond.metric)
        measured[cond.metric] = round(value, 4)
        line = cond.describe(value)
        counts = (counts_resolver(cond.metric)
                  if counts_resolver else None)
        interval = None
        n_label = ""
        prescription = None
        if isinstance(counts, tuple) and len(counts) == 2:
            k, n = counts
            if n > 0:
                interval = wilson_interval(k, n)
                n_label = "n={}".format(n)
                prescription = required_n(value, cond.threshold)
        elif isinstance(counts, tuple) and counts \
                and counts[0] == "auroc":
            _tag, a, n_pos, n_neg = counts
            if n_pos > 0 and n_neg > 0:
                interval = auroc_interval(a, n_pos, n_neg)
                n_label = "n={}+/{}-".format(n_pos, n_neg)
        if interval is not None:
            lo, hi = interval
            bar = cond.threshold
            decisive = hi < bar or lo > bar
            note = "DECISIVE" if decisive else "INCONCLUSIVE"
            extra = ""
            if not decisive and prescription is not None:
                extra = "; n~{} to resolve".format(prescription)
            line += "  [ci {:.3f}-{:.3f} {}: {}{}]".format(
                lo, hi, n_label, note, extra)
        lines.append(line)
        if not cond.holds(value):
            failed.append(cond.metric)
    return TierResult(tier="", passed=not failed,
                      measured=measured,
                      failed_conditions=failed,
                      evidence="\n".join(lines))


def _run_clean_tier(campaign, tier, solver,
                    solver_name) -> TierResult:
    pass  # intra-package import inlined above
    pass  # intra-package import inlined above
    spec = TableSpec.from_json(tier.spec_json)
    bp = plan_table(spec)
    pass  # intra-package import inlined above
    cleaned = solver([dict(r) for r in bp.dirty_rows])
    report = evaluate_cleaning(bp, cleaned, solver_name)
    tr = _judge(tier.conditions,
                lambda m: resolve_table_metric(report, m),
                lambda m: resolve_table_counts(report, m))
    tr.tier = tier.name
    return tr


def _run_predict_tier(campaign, tier, solver, solver_name,
                      offset) -> TierResult:
    pass  # intra-package import inlined above
    pass  # intra-package import inlined above
    spec = TableSpec.from_json(tier.spec_json)
    train_spec = TableSpec.from_json(tier.spec_json)
    train_spec.master_seed += offset
    train_bp = plan_table(train_spec)
    test_bp = plan_table(spec)
    outcome = campaign.outcome
    n_train = len(train_bp.clean_rows)
    train_rows = [
        {k: v for k, v in row.items() if k != outcome}
        for row in train_bp.dirty_rows[:n_train]]
    train_labels = [
        1 if train_bp.clean_rows[r][outcome] == "True" else 0
        for r in range(n_train)]
    n_test = len(test_bp.clean_rows)
    test_rows = [
        {k: v for k, v in row.items() if k != outcome}
        for row in test_bp.dirty_rows[:n_test]]
    scores = solver(train_rows, train_labels, test_rows)
    pass  # intra-package import inlined above
    report = evaluate_prediction(test_bp, outcome, list(scores),
                                 solver_name)
    tr = _judge(tier.conditions,
                lambda m: resolve_prediction_metric(report, m),
                lambda m: resolve_prediction_counts(report, m))
    tr.tier = tier.name
    # The ceiling is always reported, condition or not — it is
    # the number that gives every other number its meaning.
    tr.measured["predict.auroc"] = round(report.auroc, 4)
    tr.measured["predict.ceiling_auroc"] = round(
        report.ceiling_auroc, 4)
    tr.evidence += "\n" + report.format_text()
    return tr


def _run_regress_tier(campaign, tier, solver, solver_name,
                      offset) -> TierResult:
    """Same blinding law as predict: train on a shifted-seed
    table, score on the tier table; labels are floats."""
    pass  # intra-package import inlined above
    pass  # intra-package import inlined above
    spec = TableSpec.from_json(tier.spec_json)
    train_spec = TableSpec.from_json(tier.spec_json)
    train_spec.master_seed += offset
    train_bp = plan_table(train_spec)
    test_bp = plan_table(spec)
    outcome = campaign.outcome
    n_train = len(train_bp.clean_rows)
    train_rows = [
        {k: v for k, v in row.items() if k != outcome}
        for row in train_bp.dirty_rows[:n_train]]
    train_labels = [
        float(train_bp.clean_rows[r][outcome])
        for r in range(n_train)]
    n_test = len(test_bp.clean_rows)
    test_rows = [
        {k: v for k, v in row.items() if k != outcome}
        for row in test_bp.dirty_rows[:n_test]]
    preds = solver(train_rows, train_labels, test_rows)
    report = evaluate_regression(test_bp, outcome, list(preds),
                                 solver_name)
    tr = _judge(tier.conditions,
                lambda m: resolve_regression_metric(report, m))
    tr.tier = tier.name
    tr.measured["regress.r2"] = report.r2
    tr.measured["regress.ceiling_r2"] = report.ceiling_r2
    tr.evidence += "\n" + report.format_text()
    return tr


def _run_extract_tier(campaign, tier, solver,
                      solver_name) -> TierResult:
    pass  # intra-package import inlined above
    pass  # intra-package import inlined above
    pass  # intra-package import inlined above
    pass  # intra-package import inlined above
    pass  # intra-package import inlined above
    spec = DataSpec.from_json(tier.spec_json)
    spec.validate()
    blueprints = plan_corpus(spec)
    documents, _render_report = render_corpus(
        spec, blueprints, StubBackend())
    pass  # intra-package import inlined above
    extractor = (solver if hasattr(solver, "extract")
                 else FunctionExtractor(solver, solver_name))
    report = evaluate(blueprints, documents, extractor)
    tr = _judge(tier.conditions,
                lambda m: resolve_metric(report, m),
                lambda m: resolve_eval_counts(report, m))
    tr.tier = tier.name
    return tr


## Library: The autosolver — synthkit competes on its own data

*Source of truth: `synthkit/autosolver.py` — this cell is generated, not hand-edited.*


In [ ]:
"""SYNTH_C1: the autosolver — synthkit competes on its own data.

Because synthkit generates unlimited labeled training data, it can
train its OWN baseline and stand it next to any vendor's number:

    ceiling 0.87 / synthkit baseline 0.83 / vendor 0.71

Blinding is structural: the baseline trains on a shifted-seed
table and never sees test rows' labels or the answer key. It is a
deliberately modest model — stdlib logistic regression over
sniffed-and-normalized features — because its job is to be the
FLOOR: if a vendor cannot beat thirty lines of gradient descent
trained in seconds, the ceiling report writes the meeting summary.

Also here: `autoclean`, the heuristic reference cleaner (strip,
case-fold to column convention, unify date formats, de-format
numbers, flag exact duplicates) — the floor for cleaning
campaigns.

Deterministic end to end. Python 3.8 compatible. Stdlib only.
"""
from __future__ import annotations

import math
import re
from dataclasses import dataclass, field
from typing import Any, Callable, Dict, List, Optional, Tuple

_NUM_RE = re.compile(r"^-?\d+(\.\d+)?$")
_DATE_FORMATS = ["%Y-%m-%d", "%m/%d/%Y", "%d-%b-%Y", "%B %d, %Y",
                 "%Y.%m.%d"]
_TRUE = {"true", "yes", "y", "1"}
_FALSE = {"false", "no", "n", "0"}
_MISSING = {"", "null", "n/a", "na", "?", "none"}


def _clean_number(raw: str) -> Optional[float]:
    s = str(raw).strip().replace("$", "").replace(",", "")
    s = s.replace(" . ", ".").replace(" ", "")
    if _NUM_RE.match(s):
        return float(s)
    return None


def _clean_date(raw: str) -> Optional[float]:
    from datetime import datetime
    s = str(raw).strip()
    for fmt in _DATE_FORMATS:
        try:
            return float(datetime.strptime(s, fmt).toordinal())
        except ValueError:
            continue
    return None


def _clean_bool(raw: str) -> Optional[float]:
    s = str(raw).strip().lower()
    if s in _TRUE:
        return 1.0
    if s in _FALSE:
        return 0.0
    return None


@dataclass
class _Column:
    name: str
    kind: str                      # numeric|date|bool|categorical
    mean: float = 0.0
    std: float = 1.0
    categories: List[str] = field(default_factory=list)

    def width(self) -> int:
        return len(self.categories) if self.kind == "categorical" \
            else 1


class FeatureEncoder:
    """Sniffs column types from TRAIN rows only; encodes any rows
    into a fixed-width standardized feature vector. Deterministic;
    ignores columns starting with underscore."""

    def __init__(self, max_categories: int = 8):
        self.max_categories = max_categories
        self.columns: List[_Column] = []

    def fit(self, rows: List[Dict[str, str]]) -> "FeatureEncoder":
        if not rows:
            raise ValueError("cannot fit an encoder on zero rows")
        names = [k for k in rows[0] if not k.startswith("_")]
        for name in names:
            raw = [str(r.get(name, "")) for r in rows]
            present = [v for v in raw
                       if v.strip().lower() not in _MISSING]
            n = max(len(present), 1)
            nums = [_clean_number(v) for v in present]
            dates = [_clean_date(v) for v in present]
            bools = [_clean_bool(v) for v in present]
            num_rate = sum(1 for v in nums if v is not None) / n
            date_rate = sum(1 for v in dates if v is not None) / n
            bool_rate = sum(1 for v in bools if v is not None) / n
            if bool_rate > 0.9:
                kind, vals = "bool", [v for v in bools
                                      if v is not None]
            elif date_rate > 0.6:
                kind, vals = "date", [v for v in dates
                                      if v is not None]
            elif num_rate > 0.6:
                kind, vals = "numeric", [v for v in nums
                                         if v is not None]
            else:
                counts: Dict[str, int] = {}
                for v in present:
                    key = v.strip().lower()
                    counts[key] = counts.get(key, 0) + 1
                cats = [c for c, _cnt in sorted(
                    counts.items(),
                    key=lambda kv: (-kv[1], kv[0]))][
                        :self.max_categories]
                self.columns.append(_Column(
                    name=name, kind="categorical",
                    categories=cats))
                continue
            mean = sum(vals) / len(vals) if vals else 0.0
            var = (sum((v - mean) ** 2 for v in vals) / len(vals)
                   if vals else 0.0)
            self.columns.append(_Column(
                name=name, kind=kind, mean=mean,
                std=math.sqrt(var) or 1.0))
        return self

    def encode(self, row: Dict[str, str]) -> List[float]:
        out: List[float] = []
        for col in self.columns:
            raw = str(row.get(col.name, ""))
            if col.kind == "categorical":
                key = raw.strip().lower()
                out.extend(1.0 if key == c else 0.0
                           for c in col.categories)
                continue
            val = {"numeric": _clean_number,
                   "date": _clean_date,
                   "bool": _clean_bool}[col.kind](raw)
            if val is None:
                val = col.mean          # train-mean imputation
            out.append((val - col.mean) / col.std)
        return out


class LogisticBaseline:
    """Full-batch gradient descent, L2, fixed schedule, zero
    randomness. The floor, not the frontier."""

    def __init__(self, lr: float = 0.5, epochs: int = 300,
                 l2: float = 1e-3):
        self.lr = lr
        self.epochs = epochs
        self.l2 = l2
        self.encoder = FeatureEncoder()
        self.w: List[float] = []
        self.b = 0.0

    def fit(self, rows: List[Dict[str, str]],
            labels: List[int]) -> "LogisticBaseline":
        self.encoder.fit(rows)
        x = [self.encoder.encode(r) for r in rows]
        y = [1.0 if v else 0.0 for v in labels]
        n, d = len(x), len(x[0]) if x else 0
        self.w = [0.0] * d
        self.b = 0.0
        for _ in range(self.epochs):
            grad_w = [0.0] * d
            grad_b = 0.0
            for xi, yi in zip(x, y):
                z = self.b + sum(w * v
                                 for w, v in zip(self.w, xi))
                p = 1.0 / (1.0 + math.exp(-max(min(z, 30), -30)))
                err = p - yi
                grad_b += err
                for j, v in enumerate(xi):
                    grad_w[j] += err * v
            self.b -= self.lr * grad_b / n
            for j in range(d):
                self.w[j] -= self.lr * (
                    grad_w[j] / n + self.l2 * self.w[j])
        return self

    def score(self, rows: List[Dict[str, str]]) -> List[float]:
        out = []
        for r in rows:
            xi = self.encoder.encode(r)
            z = self.b + sum(w * v for w, v in zip(self.w, xi))
            out.append(1.0 / (1.0 + math.exp(
                -max(min(z, 30), -30))))
        return out


class LinearBaseline:
    """Ridge regression by full-batch gradient descent on the
    same mess-tolerant encoder; labels standardized for training,
    predictions returned in label units. Deterministic."""

    def __init__(self, lr: float = 0.1, epochs: int = 400,
                 l2: float = 1e-3):
        self.lr = lr
        self.epochs = epochs
        self.l2 = l2
        self.encoder = FeatureEncoder()
        self.w: List[float] = []
        self.b = 0.0
        self.y_mean = 0.0
        self.y_std = 1.0

    def fit(self, rows, labels) -> "LinearBaseline":
        self.encoder.fit(rows)
        x = [self.encoder.encode(r) for r in rows]
        n = len(x)
        self.y_mean = sum(labels) / n
        var = sum((v - self.y_mean) ** 2 for v in labels) / n
        self.y_std = math.sqrt(var) or 1.0
        y = [(v - self.y_mean) / self.y_std for v in labels]
        d = len(x[0]) if x else 0
        self.w = [0.0] * d
        self.b = 0.0
        for _ in range(self.epochs):
            grad_w = [0.0] * d
            grad_b = 0.0
            for xi, yi in zip(x, y):
                err = (self.b + sum(w * v for w, v
                                    in zip(self.w, xi))) - yi
                grad_b += err
                for j, v in enumerate(xi):
                    grad_w[j] += err * v
            self.b -= self.lr * grad_b / n
            for j in range(d):
                self.w[j] -= self.lr * (
                    grad_w[j] / n + self.l2 * self.w[j])
        return self

    def predict(self, rows) -> List[float]:
        out = []
        for r in rows:
            xi = self.encoder.encode(r)
            z = self.b + sum(w * v for w, v in zip(self.w, xi))
            out.append(z * self.y_std + self.y_mean)
        return out


def autosolver_regress(**kwargs) -> Callable:
    """The regress-campaign-contract baseline."""
    def solve(train_rows, train_labels, test_rows):
        model = LinearBaseline(**kwargs)
        model.fit(train_rows, [float(v) for v in train_labels])
        return model.predict(test_rows)
    solve.__name__ = "synthkit-baseline-regress"
    return solve


def autosolver(**kwargs) -> Callable:
    """The campaign-contract solver: train on the blinded train
    split, score the test split."""
    def solve(train_rows, train_labels, test_rows):
        model = LogisticBaseline(**kwargs)
        model.fit(train_rows, train_labels)
        return model.score(test_rows)
    solve.__name__ = "synthkit-baseline"
    return solve


# ===================================================================
# autoclean — the heuristic reference cleaner
# ===================================================================

def autoclean(rows: List[Dict[str, str]]
              ) -> List[Dict[str, str]]:
    """Normalizes what heuristics can honestly normalize:
    whitespace, casing (to each column's dominant convention),
    date formats (to ISO), numeric formatting ($, thousands
    separators, spaced decimals); flags exact duplicate rows.
    Never guesses at missing or wrong values — that honesty keeps
    its overcorrection near zero."""
    from datetime import datetime
    if not rows:
        return []
    names = [k for k in rows[0] if not k.startswith("_")]
    case_mode: Dict[str, str] = {}
    for name in names:
        tally = {"lower": 0, "upper": 0, "title": 0}
        for r in rows:
            v = str(r.get(name, "")).strip()
            if not v or _clean_number(v) is not None:
                continue
            if v == v.lower():
                tally["lower"] += 1
            elif v == v.upper():
                tally["upper"] += 1
            elif v == v.title():
                tally["title"] += 1
        case_mode[name] = max(tally, key=lambda k: tally[k]) \
            if any(tally.values()) else "none"

    out: List[Dict[str, str]] = []
    seen: Dict[Tuple, int] = {}
    for r in rows:
        row: Dict[str, str] = {}
        for name in names:
            v = str(r.get(name, "")).strip()
            if v.strip().lower() in _MISSING:
                row[name] = v
                continue
            d = _clean_date(v)
            if d is not None:
                row[name] = datetime.fromordinal(
                    int(d)).date().isoformat()
                continue
            num = _clean_number(v)
            if num is not None:
                row[name] = ("{:.4f}".format(num)
                             .rstrip("0").rstrip(".")
                             if "." in repr(num) else str(num))
                if num == int(num):
                    row[name] = str(int(num))
                continue
            b = _clean_bool(v)
            if b is not None and v.lower() not in ("true",
                                                   "false"):
                row[name] = "True" if b else "False"
                continue
            mode = case_mode.get(name, "none")
            if mode == "lower":
                row[name] = v.lower()
            elif mode == "upper":
                row[name] = v.upper()
            elif mode == "title":
                row[name] = v.title()
            else:
                row[name] = v
        key = tuple(row[n] for n in names)
        if key in seen:
            row["_duplicate"] = "true"
        seen[key] = seen.get(key, 0) + 1
        out.append(row)
    return out


# ===================================================================
# The showdown
# ===================================================================

@dataclass
class ShowdownTier:
    tier: str
    ceiling: float
    baseline_auroc: float
    vendor_auroc: float
    vendor_passed: bool

    @property
    def verdict(self) -> str:
        if abs(self.vendor_auroc - self.baseline_auroc) < 1e-9:
            return "vendor matches the baseline"
        if self.vendor_auroc > self.baseline_auroc:
            return "vendor beats the baseline"
        return "vendor loses to a stdlib baseline"


@dataclass
class ShowdownResult:
    title: str
    vendor_name: str
    tiers: List[ShowdownTier]

    def format_text(self) -> str:
        lines = ["SHOWDOWN: {} vs synthkit baseline — {}".format(
            self.vendor_name, self.title)]
        for t in self.tiers:
            lines.append(
                "  {:<14} ceiling {:.3f} / baseline {:.3f} / "
                "{} {:.3f}  [{}] {}".format(
                    t.tier, t.ceiling, t.baseline_auroc,
                    self.vendor_name, t.vendor_auroc,
                    "PASS" if t.vendor_passed else "FAIL",
                    t.verdict))
        return "\n".join(lines)


def run_showdown(campaign, vendor: Callable,
                 vendor_name: str = "vendor",
                 train_seed_offset: int = 1000) -> ShowdownResult:
    pass  # intra-package import inlined above
    pass  # intra-package import inlined above
    pass  # intra-package import inlined above
    pass  # intra-package import inlined above
    if campaign.goal != "predict":
        raise ValueError("showdowns run on predict campaigns")
    baseline = autosolver()
    tiers: List[ShowdownTier] = []
    for tier in campaign.tiers:
        spec = TableSpec.from_json(tier.spec_json)
        train_spec = TableSpec.from_json(tier.spec_json)
        train_spec.master_seed += train_seed_offset
        train_bp = plan_table(train_spec)
        test_bp = plan_table(spec)
        outcome = campaign.outcome
        n_train = len(train_bp.clean_rows)
        n_test = len(test_bp.clean_rows)
        train_rows = [
            {k: v for k, v in row.items() if k != outcome}
            for row in train_bp.dirty_rows[:n_train]]
        train_labels = [
            1 if train_bp.clean_rows[r][outcome] == "True" else 0
            for r in range(n_train)]
        test_rows = [
            {k: v for k, v in row.items() if k != outcome}
            for row in test_bp.dirty_rows[:n_test]]
        v_scores = list(vendor(train_rows, train_labels,
                               test_rows))
        b_scores = baseline(train_rows, train_labels, test_rows)
        v_rep = evaluate_prediction(test_bp, outcome, v_scores,
                                    vendor_name)
        b_rep = evaluate_prediction(test_bp, outcome, b_scores,
                                    "synthkit-baseline")
        tr = _judge(tier.conditions,
                    lambda m: resolve_prediction_metric(v_rep, m))
        tiers.append(ShowdownTier(
            tier=tier.name,
            ceiling=round(v_rep.ceiling_auroc, 4),
            baseline_auroc=round(b_rep.auroc, 4),
            vendor_auroc=round(v_rep.auroc, 4),
            vendor_passed=tr.passed,
        ))
    return ShowdownResult(title=campaign.title,
                          vendor_name=vendor_name, tiers=tiers)


## Library: Semantic lint — does the spec mean what you meant?

*Source of truth: `synthkit/lint.py` — this cell is generated, not hand-edited.*


In [ ]:
"""SYNTH_L1: semantic lint — the checks a validator cannot make.

Validation answers "is this spec lawful?"; lint answers "will this
spec produce what you MEANT?" by planning a deterministic probe
and inspecting the realized data. Every check here was a human
catch first, at station 03, squinting at a preview:

  L1 outcome prevalence      (a live -1.4 intercept yielded 71%
                              against a requested 15-20%)
  L2 derived-value sanity    (a dollar-amount noise_sigma made
                              10^150-dollar hospital stays)
  L3 rule-coupling truth     (an edit dropped a date rule and
                              gaps silently decoupled from stays)
  L4 fossil distributions    (rule targets carrying ignored
                              distributions mislead readers)
  L5 ignored name pools      (person_name synthesizes; provided
                              choices are silently unused)
  L6 description coverage    (the English asked for outliers;
                              the spec forgot them)

Findings are WARN (probably not what you meant) or INFO (worth
knowing); ERRORs belong to validation. `target_prevalence` on an
outcome makes L1 enforceable: "around 15-20 percent" in English
becomes [0.15, 0.20] in the spec becomes a measured check.

Python 3.8 compatible. Stdlib only.
"""
from __future__ import annotations

import math
from dataclasses import dataclass, field
from typing import List, Optional, Tuple

pass  # intra-package import inlined above

_MESS_KEYWORDS = [
    ("duplicate", "duplicate_rate",
     lambda spec: spec.duplicate_rate > 0),
    ("missing", "missing_rate",
     lambda spec: any(c.mess.missing_rate > 0
                      for c in spec.columns)),
    ("outlier", "outlier_rate",
     lambda spec: any(c.mess.outlier_rate > 0
                      for c in spec.columns)),
    ("typo", "typo_rate",
     lambda spec: any(c.mess.typo_rate > 0
                      for c in spec.columns)),
    ("wrong", "wrong_rate",
     lambda spec: any(c.mess.wrong_rate > 0
                      for c in spec.columns)),
    ("casing", "case_rate",
     lambda spec: any(c.mess.case_rate > 0
                      for c in spec.columns)),
    ("whitespace", "space_rate",
     lambda spec: any(c.mess.space_rate > 0
                      for c in spec.columns)),
]


@dataclass
class LintFinding:
    level: str          # WARN | INFO
    code: str
    message: str


@dataclass
class LintReport:
    findings: List[LintFinding] = field(default_factory=list)
    probe_rows: int = 0

    @property
    def ok(self) -> bool:
        return not any(f.level == "WARN" for f in self.findings)

    def format_text(self) -> str:
        if not self.findings:
            return ("SEMANTIC LINT: clean ({} probe rows) — the "
                    "spec appears to mean what it says."
                    .format(self.probe_rows))
        lines = ["SEMANTIC LINT ({} probe rows): {} finding(s)"
                 .format(self.probe_rows, len(self.findings))]
        for f in self.findings:
            lines.append("  {:<4} [{}] {}".format(
                f.level, f.code, f.message))
        return "\n".join(lines)


def lint_table(spec: TableSpec, description: str = "",
               probe_rows: int = 300) -> LintReport:
    """Plan a deterministic probe and inspect what the spec
    actually produces. Never raises for content reasons; the spec
    must already validate."""
    pass  # intra-package import inlined above
    spec.validate()
    probe = TableSpec.from_json(spec.to_json())
    probe.rows = min(spec.rows, probe_rows)
    bp = plan_table(probe)
    n = len(bp.clean_rows)
    report = LintReport(probe_rows=n)
    add = report.findings.append

    rule_targets = {r.get("later") for r in spec.rules
                    if r.get("kind") == "date_after"}
    rule_targets |= {r.get("target") for r in spec.rules
                     if r.get("kind") == "derived"}

    # ---- L1: outcome prevalence ----
    for oc in spec.outcomes:
        name = oc["name"]
        if oc.get("kind") == "linear":
            vals = [float(r[name]) for r in bp.clean_rows]
            mean = sum(vals) / n
            var = sum((v - mean) ** 2 for v in vals) / n
            tr_ = oc.get("target_range")
            if var < 1e-12:
                add(LintFinding(
                    "WARN", "L1-degenerate",
                    "outcome `{}` has zero variance — no signal "
                    "to regress".format(name)))
            elif tr_ and not (float(tr_[0]) <= mean
                              <= float(tr_[1])):
                add(LintFinding(
                    "WARN", "L1-range",
                    "outcome `{}` realizes mean {:.2f} against "
                    "the declared target range [{}, {}] — "
                    "adjust the intercept".format(
                        name, mean, tr_[0], tr_[1])))
            else:
                add(LintFinding(
                    "INFO", "L1-range",
                    "outcome `{}` realizes mean {:.2f}, sd "
                    "{:.2f}".format(name, mean, var ** 0.5)))
            continue
        rate = sum(1 for r in bp.clean_rows
                   if r[name] == "True") / n
        target = oc.get("target_prevalence")
        if target:
            lo, hi = float(target[0]), float(target[1])
            if not (lo <= rate <= hi):
                add(LintFinding(
                    "WARN", "L1-prevalence",
                    "outcome `{}` realizes {:.1%} against the "
                    "declared target [{:.0%}, {:.0%}] — adjust "
                    "the intercept (more negative = rarer)"
                    .format(name, rate, lo, hi)))
            else:
                add(LintFinding(
                    "INFO", "L1-prevalence",
                    "outcome `{}` realizes {:.1%}, inside its "
                    "declared target".format(name, rate)))
        elif rate < 0.01 or rate > 0.9:
            add(LintFinding(
                "WARN", "L1-degenerate",
                "outcome `{}` realizes {:.1%} — nearly single-"
                "class; prediction campaigns will be vacuous"
                .format(name, rate)))
        else:
            add(LintFinding(
                "INFO", "L1-prevalence",
                "outcome `{}` realizes {:.1%} (no declared "
                "target — add `target_prevalence: [lo, hi]` to "
                "enforce)".format(name, rate)))

    # ---- L2: derived-value sanity ----
    for rule in spec.rules:
        if rule.get("kind") != "derived":
            continue
        target = rule["target"]
        vals = []
        for r in bp.clean_rows:
            try:
                vals.append(float(r[target]))
            except ValueError:
                pass
        if not vals:
            continue
        top = max(abs(v) for v in vals)
        if not all(math.isfinite(v) for v in vals) or top > 1e12:
            add(LintFinding(
                "WARN", "L2-derived",
                "derived column `{}` reaches magnitude {:.2e} — "
                "check `factor` and `noise_sigma` (scale belongs "
                "in factor)".format(target, top)))
        elif all(v == 0 for v in vals):
            add(LintFinding(
                "WARN", "L2-derived",
                "derived column `{}` is all zeros — factor or "
                "source likely wrong".format(target)))

    # ---- L3: rule-coupling truth ----
    from datetime import date as _date
    for rule in spec.rules:
        if rule.get("kind") != "date_after" \
                or not rule.get("days_from"):
            continue
        later, earlier = rule["later"], rule["earlier"]
        src = rule["days_from"]
        bad = 0
        for r in bp.clean_rows:
            gap = (_date.fromisoformat(r[later])
                   - _date.fromisoformat(r[earlier])).days
            if gap != max(int(round(float(r[src]))), 0):
                bad += 1
        if bad:
            add(LintFinding(
                "WARN", "L3-coupling",
                "date gap `{}`-`{}` disagrees with `{}` in "
                "{}/{} probe rows — rule ordering or an "
                "overriding rule is interfering".format(
                    later, earlier, src, bad, n)))

    # ---- L4: fossil distributions ----
    for col in spec.columns:
        if col.name in rule_targets and col.distribution:
            add(LintFinding(
                "INFO", "L4-fossil",
                "column `{}` carries a distribution that its "
                "rule overwrites — omit it to say what you mean"
                .format(col.name)))

    # ---- L5: ignored name pools ----
    for col in spec.columns:
        if col.ctype == "person_name" and col.distribution:
            add(LintFinding(
                "INFO", "L5-names",
                "column `{}`: person_name synthesizes from a "
                "built-in pool; the provided distribution is "
                "ignored".format(col.name)))

    # ---- L6: description coverage ----
    if description:
        text = description.lower()
        for keyword, knob, present in _MESS_KEYWORDS:
            if keyword in text and not present(spec):
                add(LintFinding(
                    "WARN", "L6-coverage",
                    "the description mentions `{}` but no "
                    "column sets {} — a mess clause was dropped"
                    .format(keyword, knob)))
    return report


# ===================================================================
# Corpus lint — the document side of "did you mean this?"
# ===================================================================

def lint_corpus(spec, description: str = "",
                probe_docs: int = 30) -> LintReport:
    """Plans a deterministic probe corpus and inspects what the
    document spec actually plants."""
    pass  # intra-package import inlined above
    pass  # intra-package import inlined above
    assert isinstance(spec, DataSpec)
    spec.validate()
    probe = DataSpec.from_json(spec.to_json())
    probe.corpus.size = min(spec.corpus.size, probe_docs)
    blueprints = plan_corpus(probe)
    n = len(blueprints)
    report = LintReport(probe_rows=n)
    add = report.findings.append

    planted: dict = {}
    distracted: dict = {}
    styles: dict = {}
    for bp in blueprints:
        for note in bp.notes:
            for el in note.elements:
                planted[el.element_id] = planted.get(
                    el.element_id, 0) + 1
            for d in note.distractors:
                distracted[d.distractor_id] = distracted.get(
                    d.distractor_id, 0) + 1
            for axis, val in note.style.items():
                styles.setdefault(axis, set()).add(val)

    element_ids = set()
    distractor_ids = set()
    for uf in spec.unstructured_fields:
        for el in uf.target_elements:
            element_ids.add(el.element_id)
            if planted.get(el.element_id, 0) == 0:
                add(LintFinding(
                    "WARN", "D1-never-planted",
                    "element `{}` was never planted across {} "
                    "probe documents — its recall will divide "
                    "by zero conceptually; check probability "
                    "and difficulty".format(el.element_id, n)))
        for dis in uf.distractors:
            distractor_ids.add(dis.distractor_id)
            if dis.density > 0 and distracted.get(
                    dis.distractor_id, 0) == 0:
                add(LintFinding(
                    "WARN", "D2-toothless",
                    "distractor `{}` has density {} but landed "
                    "in zero probe documents — the trap is "
                    "unloaded".format(dis.distractor_id,
                                      dis.density)))

    overlap = element_ids & distractor_ids
    if overlap:
        add(LintFinding(
            "WARN", "D3-collision",
            "ids used as BOTH element and distractor: {} — "
            "scoring cannot tell reward from trap".format(
                ", ".join(sorted(overlap)))))

    for axis, vals in sorted(styles.items()):
        if len(vals) == 1:
            add(LintFinding(
                "INFO", "D4-flat-style",
                "style axis `{}` drew a single value across "
                "the probe — the corpus will read uniformly"
                .format(axis)))

    if description:
        text = description.lower()
        if ("trap" in text or "distractor" in text) \
                and not distractor_ids:
            add(LintFinding(
                "WARN", "D5-coverage",
                "the description asks for traps/distractors "
                "but the spec defines none"))
    return report


## Library: Relational tables — join mess with an answer key

*Source of truth: `synthkit/relational.py` — this cell is generated, not hand-edited.*


In [ ]:
"""SYNTH_R1: relational tables — multi-table truth with join
mess.

Real hospital data is never one table: encounters reference
patients, labs reference encounters, and the mess that matters
most lives in the JOINS — orphaned foreign keys pointing at
parents that do not exist. No flat-table product can test a
vendor's referential-integrity claims; a generator that plants
orphans and keeps the answer key can.

Design:
  - A RelationalSpec holds named TableSpecs plus links. Each link
    declares child/parent tables, the parent's key column (must be
    a `sequence` str_id — guaranteed unique), and the fk column to
    INJECT into the child (it does not appear in the child's own
    spec; the link owns it).
  - Planning is layered determinism: each table plans under its
    own derived seed; fk values are drawn from the parent's real
    keys by seeded RNG; then `orphan_rate` replaces a seeded
    fraction of DIRTY fk cells with format-valid keys that do not
    exist — every injection ledgered in a LinkLedger. Clean rows
    keep true references always.
  - evaluate_links scores an orphan-flagger (a vendor claiming to
    detect referential breaks) with precision/recall against the
    ledger.

Python 3.8 compatible. Stdlib only.
"""
from __future__ import annotations

import hashlib
import json
import random
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional

pass  # intra-package import inlined above
pass  # intra-package import inlined above


class RelationalSpecError(ValueError):
    pass


class RelationalIntegrityError(ValueError):
    pass


@dataclass
class Link:
    """orphan_style:
      'random' — fake keys far from any real one (a set-membership
                 check catches them; the easy tier)
      'near'   — fake keys one digit-transposition from a REAL key
                 (still absent from the key set, but they defeat
                 fuzzy matchers that 'repair' near-misses)
    drift_rate — the PRECISION trap: valid references whose
      formatting is mangled (case flips, stray whitespace); a
      naive set-membership checker false-flags every one."""
    child: str
    parent: str
    parent_key: str
    fk_column: str
    orphan_rate: float = 0.0
    orphan_style: str = "random"
    drift_rate: float = 0.0


@dataclass
class RelationalSpec:
    title: str
    master_seed: int
    tables: Dict[str, TableSpec]
    links: List[Link] = field(default_factory=list)

    def to_json(self) -> str:
        return json.dumps({
            "title": self.title,
            "master_seed": self.master_seed,
            "tables": {name: json.loads(t.to_json())
                       for name, t in self.tables.items()},
            "links": [{"child": l.child, "parent": l.parent,
                       "parent_key": l.parent_key,
                       "fk_column": l.fk_column,
                       "orphan_rate": l.orphan_rate,
                       "orphan_style": l.orphan_style,
                       "drift_rate": l.drift_rate}
                      for l in self.links],
        }, indent=2)

    @classmethod
    def from_json(cls, raw: str) -> "RelationalSpec":
        d = json.loads(raw)
        return cls(
            title=d.get("title", ""),
            master_seed=int(d.get("master_seed", 0)),
            tables={name: TableSpec.from_json(json.dumps(t))
                    for name, t in d.get("tables", {}).items()},
            links=[Link(child=l["child"], parent=l["parent"],
                        parent_key=l["parent_key"],
                        fk_column=l["fk_column"],
                        orphan_rate=float(
                            l.get("orphan_rate", 0.0)),
                        orphan_style=l.get("orphan_style",
                                           "random"),
                        drift_rate=float(
                            l.get("drift_rate", 0.0)))
                   for l in d.get("links", [])],
        )

    def validate(self) -> None:
        problems: List[str] = []
        if not self.tables:
            problems.append("a relational spec needs tables")
        for name, table in self.tables.items():
            try:
                table.validate()
            except Exception as e:
                problems.append(
                    "table `{}`: {}".format(name, e))
        for i, link in enumerate(self.links):
            tag = "link #{}".format(i + 1)
            if link.child not in self.tables:
                problems.append("{}: unknown child table `{}`"
                                .format(tag, link.child))
            if link.parent not in self.tables:
                problems.append("{}: unknown parent table `{}`"
                                .format(tag, link.parent))
                continue
            parent = self.tables[link.parent]
            key_cols = [c for c in parent.columns
                        if c.name == link.parent_key]
            if not key_cols:
                problems.append(
                    "{}: parent `{}` has no column `{}`".format(
                        tag, link.parent, link.parent_key))
            elif key_cols[0].ctype != "str_id" or \
                    key_cols[0].dist_kind() != "sequence":
                problems.append(
                    "{}: parent key `{}` must be a str_id "
                    "sequence column — uniqueness is the whole "
                    "point of a key".format(
                        tag, link.parent_key))
            if link.child in self.tables and any(
                    c.name == link.fk_column
                    for c in self.tables[link.child].columns):
                problems.append(
                    "{}: fk column `{}` must NOT appear in the "
                    "child spec — the link injects it".format(
                        tag, link.fk_column))
            if not (0.0 <= link.orphan_rate <= 0.5):
                problems.append(
                    "{}: orphan_rate must be in [0, 0.5]"
                    .format(tag))
            if link.orphan_style not in ("random", "near"):
                problems.append(
                    "{}: orphan_style must be `random` or "
                    "`near`".format(tag))
            if not (0.0 <= link.drift_rate <= 0.5):
                problems.append(
                    "{}: drift_rate must be in [0, 0.5]"
                    .format(tag))
        if problems:
            raise RelationalSpecError(
                "{} problem(s):\n".format(len(problems))
                + "\n".join("  - " + p for p in problems))


@dataclass
class OrphanMess:
    child: str
    row: int
    fk_column: str
    true_key: str
    orphan_key: str


@dataclass
class DriftMess:
    child: str
    row: int
    fk_column: str
    true_key: str
    drifted: str


@dataclass
class RelationalBlueprint:
    blueprints: Dict[str, TableBlueprint]
    link_ledger: List[OrphanMess] = field(default_factory=list)
    drift_ledger: List[DriftMess] = field(default_factory=list)


def _table_seed(master_seed: int, name: str) -> int:
    digest = hashlib.sha256(
        "{}:{}".format(master_seed, name).encode()).hexdigest()
    return int(digest[:12], 16)


def plan_relational(spec: RelationalSpec) -> RelationalBlueprint:
    spec.validate()
    blueprints: Dict[str, TableBlueprint] = {}
    for name, table in spec.tables.items():
        derived = TableSpec.from_json(table.to_json())
        derived.master_seed = _table_seed(spec.master_seed, name)
        blueprints[name] = plan_table(derived)
    ledger: List[OrphanMess] = []
    drift: List[DriftMess] = []
    for link in spec.links:
        parent_bp = blueprints[link.parent]
        child_bp = blueprints[link.child]
        parent_keys = [r[link.parent_key]
                       for r in parent_bp.clean_rows]
        prefix = "".join(
            ch for ch in parent_keys[0]
            if not ch.isdigit()) if parent_keys else "X"
        width = len(parent_keys[0]) - len(prefix) \
            if parent_keys else 5
        rng = random.Random(_table_seed(
            spec.master_seed,
            "link:{}:{}".format(link.child, link.fk_column)))
        n_clean = len(child_bp.clean_rows)
        assignments = [rng.choice(parent_keys)
                       for _ in range(n_clean)]
        for i, row in enumerate(child_bp.clean_rows):
            row[link.fk_column] = assignments[i]
        # Dirty rows include duplicates; map each dirty row back
        # to its clean origin via the trailing append order: the
        # first n_clean dirty rows correspond 1:1, appended
        # duplicates copy their source row's assignment.
        for i, row in enumerate(child_bp.dirty_rows):
            src_idx = i if i < n_clean else \
                _dup_source(child_bp, i, n_clean)
            row[link.fk_column] = assignments[src_idx]
        # Join mess on DIRTY only, seeded per row; orphan and
        # drift are mutually exclusive per cell (orphan first —
        # a broken reference is the deeper corruption).
        existing = set(parent_keys)
        for i in range(len(child_bp.dirty_rows)):
            src_idx = i if i < n_clean else \
                _dup_source(child_bp, i, n_clean)
            true_key = assignments[src_idx]
            r = random.Random(_table_seed(
                spec.master_seed,
                "orphan:{}:{}:{}".format(
                    link.child, link.fk_column, i)))
            if r.random() < link.orphan_rate:
                fake = _make_orphan(link.orphan_style, true_key,
                                    prefix, width, existing, r)
                if fake is None:
                    continue
                ledger.append(OrphanMess(
                    child=link.child, row=i,
                    fk_column=link.fk_column,
                    true_key=true_key, orphan_key=fake))
                child_bp.dirty_rows[i][link.fk_column] = fake
                continue
            rd = random.Random(_table_seed(
                spec.master_seed,
                "drift:{}:{}:{}".format(
                    link.child, link.fk_column, i)))
            if rd.random() < link.drift_rate:
                mangled = _drift_key(true_key, rd)
                if mangled == true_key:
                    continue
                drift.append(DriftMess(
                    child=link.child, row=i,
                    fk_column=link.fk_column,
                    true_key=true_key, drifted=mangled))
                child_bp.dirty_rows[i][link.fk_column] = mangled
        if link.fk_column not in child_bp.columns:
            child_bp.columns.append(link.fk_column)
    return RelationalBlueprint(blueprints=blueprints,
                               link_ledger=ledger,
                               drift_ledger=drift)


def _make_orphan(style: str, true_key: str, prefix: str,
                 width: int, existing, r) -> Optional[str]:
    if style == "near":
        digits = list(true_key[len(prefix):])
        for _attempt in range(20):
            if len(digits) < 2:
                break
            j = r.randint(0, len(digits) - 2)
            swapped = list(digits)
            swapped[j], swapped[j + 1] = \
                swapped[j + 1], swapped[j]
            cand = prefix + "".join(swapped)
            if cand not in existing and cand != true_key:
                return cand
        return None
    for _attempt in range(20):
        cand = "{}{:0{}d}".format(
            prefix, r.randint(10 ** width,
                              2 * 10 ** width - 1), width)
        if cand not in existing:
            return cand
    return None


def _drift_key(key: str, r) -> str:
    choice = r.randint(0, 2)
    if choice == 0:
        return key.lower()
    if choice == 1:
        return " " + key
    return key + "  "


def _dup_source(bp: TableBlueprint, dirty_idx: int,
                n_clean: int) -> int:
    """Appended duplicates carry their source row index when the
    planner recorded one; fall back to a stable wrap."""
    dup_map = getattr(bp, "duplicate_of", None)
    if isinstance(dup_map, dict) and dirty_idx in dup_map:
        return dup_map[dirty_idx]
    return dirty_idx % max(n_clean, 1)


@dataclass
class LinkReport:
    link: str
    orphans_planted: int
    orphans_flagged: int
    false_flags: int
    n_rows: int
    drift_planted: int = 0
    drift_false_flagged: int = 0

    @property
    def recall(self) -> float:
        return (self.orphans_flagged / self.orphans_planted
                if self.orphans_planted else 0.0)

    @property
    def precision(self) -> float:
        total = self.orphans_flagged + self.false_flags
        return self.orphans_flagged / total if total else 0.0

    def format_text(self) -> str:
        text = ("LINK {}: {} orphan(s) planted in {} rows — "
                "recall {:.3f}, precision {:.3f}".format(
                    self.link, self.orphans_planted,
                    self.n_rows, self.recall, self.precision))
        if self.drift_planted:
            text += ("; {} drifted-but-VALID ref(s), {} "
                     "false-flagged".format(
                         self.drift_planted,
                         self.drift_false_flagged))
        return text


def evaluate_links(rbp: RelationalBlueprint, link: Link,
                   flags: List[bool],
                   flagger_name: str = "flagger") -> LinkReport:
    """Scores a vendor's orphan flags (one bool per dirty child
    row) against the ledger."""
    child_bp = rbp.blueprints[link.child]
    n = len(child_bp.dirty_rows)
    if len(flags) != n:
        raise RelationalIntegrityError(
            "flags must cover every dirty child row: got {} "
            "for {}".format(len(flags), n))
    planted = {m.row for m in rbp.link_ledger
               if m.child == link.child
               and m.fk_column == link.fk_column}
    drifted = {m.row for m in rbp.drift_ledger
               if m.child == link.child
               and m.fk_column == link.fk_column}
    flagged = {i for i, f in enumerate(flags) if f}
    return LinkReport(
        link="{}.{} -> {}.{}".format(
            link.child, link.fk_column, link.parent,
            link.parent_key),
        orphans_planted=len(planted),
        orphans_flagged=len(planted & flagged),
        false_flags=len(flagged - planted),
        n_rows=n,
        drift_planted=len(drifted),
        drift_false_flagged=len(drifted & flagged))


def write_relational(run_dir, spec: RelationalSpec,
                     rbp: RelationalBlueprint) -> Path:
    pass  # intra-package import inlined above
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    hashes: Dict[str, str] = {}
    for name, table in spec.tables.items():
        derived = TableSpec.from_json(table.to_json())
        derived.master_seed = _table_seed(spec.master_seed, name)
        write_table(run_dir / name, derived,
                    rbp.blueprints[name])
        sub = (run_dir / name / "manifest.json").read_text(
            encoding="utf-8")
        hashes["{}/manifest.json".format(name)] = \
            hashlib.sha256(sub.encode()).hexdigest()
    links_payload = json.dumps({
        "spec": json.loads(spec.to_json()),
        "orphans": [{"child": m.child, "row": m.row,
                     "fk_column": m.fk_column,
                     "true_key": m.true_key,
                     "orphan_key": m.orphan_key}
                    for m in rbp.link_ledger],
        "drift": [{"child": m.child, "row": m.row,
                   "fk_column": m.fk_column,
                   "true_key": m.true_key,
                   "drifted": m.drifted}
                  for m in rbp.drift_ledger],
    }, indent=2)
    (run_dir / "links.json").write_text(links_payload,
                                        encoding="utf-8")
    hashes["links.json"] = hashlib.sha256(
        links_payload.encode()).hexdigest()
    (run_dir / "manifest.json").write_text(json.dumps({
        "synthkit_relational_manifest": 1,
        "hashes": hashes,
    }, indent=2), encoding="utf-8")
    return run_dir


def load_relational(run_dir) -> dict:
    run_dir = Path(run_dir)
    manifest = json.loads(
        (run_dir / "manifest.json").read_text(encoding="utf-8"))
    for rel, expected in manifest["hashes"].items():
        actual = hashlib.sha256(
            (run_dir / rel).read_text(
                encoding="utf-8").encode()).hexdigest()
        if actual != expected:
            raise RelationalIntegrityError(
                "`{}` does not match its manifest hash — the "
                "artifact has been altered".format(rel))
    return json.loads(
        (run_dir / "links.json").read_text(encoding="utf-8"))


## Library: Examples — reference document vertical, reference table, naive extractor and cleaner

*Source of truth: `synthkit/examples.py` — this cell is generated, not hand-edited.*


In [ ]:
"""SYNTH_V1 examples: the reference vertical, a real (naive) regex
extractor, and a canned experiment — everything needed to watch the
whole machine turn over.

reference_spec() is the v1 vertical: inpatient progress notes with
current medications, follow-ups, hard-buried allergies, and a
discontinued-medication distractor trap.

regex_extract() is a deliberately naive but REAL extractor — the
kind of thing a vendor demo might hide behind an API. Its flaws are
the point: it greps medication names anywhere (so it falls into the
discontinued trap), it only knows some phrasings (so recall varies
by surface form), and it has no notion of allergy context beyond one
pattern. Running it through the harness produces an honest, sliced
failure report — the product's output, live.

Replace regex_extract with an adapter around any outside model and
nothing else changes.
"""
from __future__ import annotations

import re
from typing import List

pass  # intra-package import inlined above
pass  # intra-package import inlined above
pass  # intra-package import inlined above

MED_POOL = [
    "metformin 500mg BID",
    "lisinopril 10mg daily",
    "atorvastatin 40mg nightly",
    "levothyroxine 75mcg qAM",
]
STOPPED_POOL = ["ibuprofen", "omeprazole"]
ALLERGY_POOL = ["penicillin", "sulfa", "latex"]
FOLLOWUP_POOL = ["in two weeks", "in one month", "next Tuesday"]


def reference_spec(size: int = 50, master_seed: int = 42) -> DataSpec:
    return DataSpec(
        title="Progress note extraction test corpus",
        structured_fields=[
            StructuredField(
                name="encounter_id", ftype="id",
                distribution=Distribution(
                    kind="sequence",
                    params={"prefix": "ENC-", "start": 10000}),
            ),
            StructuredField(
                name="patient_name", ftype="person_name",
                distribution=Distribution(kind="categorical",
                                          params={"choices": ["x"]}),
            ),
            StructuredField(
                name="admit_date", ftype="date",
                distribution=Distribution(
                    kind="date_range",
                    params={"start": "2026-01-01",
                            "end": "2026-06-30"}),
            ),
            StructuredField(
                name="discharge_date", ftype="date",
                distribution=Distribution(
                    kind="date_range",
                    params={"start": "2026-01-01",
                            "end": "2026-06-30"}),
            ),
            StructuredField(
                name="length_of_stay_days", ftype="int",
                distribution=Distribution(
                    kind="lognormal",
                    params={"mu": 1.2, "sigma": 0.6,
                            "min": 1, "max": 45}),
            ),
        ],
        unstructured_fields=[
            UnstructuredField(
                name="progress_note",
                note_type="inpatient progress note",
                target_elements=[
                    TargetElement(
                        element_id="current_medication",
                        description="an active medication with dose",
                        phrasings=[
                            "continues {value} at current dose",
                            "pt remains on {value}",
                            "{value} — no changes today",
                        ],
                        density=0.9, difficulty="easy",
                        value_source={"choices": list(MED_POOL)},
                    ),
                    TargetElement(
                        element_id="followup_appointment",
                        description="a scheduled follow-up",
                        phrasings=[
                            "follow up scheduled for {value}",
                            "will see clinic again {value}",
                        ],
                        density=0.6, difficulty="medium",
                        value_source={"choices": list(FOLLOWUP_POOL)},
                    ),
                    TargetElement(
                        element_id="allergy_flag",
                        description="a documented allergy",
                        phrasings=[
                            "allergies: {value}",
                            "known {value} allergy noted",
                        ],
                        density=0.4, difficulty="hard",
                        value_source={"choices": list(ALLERGY_POOL)},
                    ),
                ],
                distractors=[
                    Distractor(
                        distractor_id="discontinued_medication",
                        description="a med explicitly STOPPED — must "
                                    "not be extracted as current",
                        phrasings=[
                            "{value} discontinued this admission",
                            "stopped {value} due to side effects",
                        ],
                        density=0.5,
                        value_source={"choices": list(STOPPED_POOL)},
                    ),
                ],
                style=StyleAxes(
                    personas=["attending", "resident", "nurse"],
                ),
                length_words=[60, 180],
            ),
        ],
        cross_field_rules=[
            CrossFieldRule(earlier="admit_date",
                           later="discharge_date",
                           min_delta=1, max_delta=21),
        ],
        corpus=CorpusConfig(size=size, master_seed=master_seed),
    )


def reference_rules() -> dict:
    return {
        "current_medication": MatchRule(
            mode="value", categories=["current", "active med"]),
        "followup_appointment": MatchRule(mode="value"),
        "allergy_flag": MatchRule(mode="value",
                                  categories=["allerg"]),
        "discontinued_medication": MatchRule(
            mode="value", categories=["current", "active med"]),
    }


# ===================================================================
# The naive-but-real regex extractor
# ===================================================================

_MED_RE = re.compile(
    r"\b((?:metformin|lisinopril|atorvastatin|levothyroxine|"
    r"ibuprofen|omeprazole)[^.\n]*?)(?:[.\n]|$)", re.IGNORECASE)
_FOLLOWUP_RE = re.compile(
    r"follow(?:\s|-)?up (?:scheduled )?for ([^.\n]+)", re.IGNORECASE)
_ALLERGY_RE = re.compile(
    r"allerg(?:y|ies)[:\s]+([^.\n]+)", re.IGNORECASE)


def regex_extract(doc_id: str, text: str) -> List[Extraction]:
    """Deliberately naive: any known med name anywhere becomes a
    'current medication' (the trap), only one follow-up phrasing is
    known, only one allergy pattern is known."""
    out: List[Extraction] = []
    for m in _MED_RE.finditer(text):
        out.append(Extraction(category="current medication",
                              text=m.group(1).strip()))
    for m in _FOLLOWUP_RE.finditer(text):
        out.append(Extraction(category="follow-up",
                              text=m.group(1).strip()))
    for m in _ALLERGY_RE.finditer(text):
        out.append(Extraction(category="allergy",
                              text=m.group(1).strip()))
    return out


def allergy_bar_experiment(backend=None, size: int = 16,
                           extractor_fn=None) -> Experiment:
    """The canned live experiment: does the extractor clear an 85%
    allergy-recall bar and stay under a 10% distractor trap rate?
    (The naive regex extractor does neither — by design.)"""
    pass  # intra-package import inlined above
    return Experiment(
        name="allergy recall bar + discontinued-med trap",
        spec=reference_spec(size=size),
        extractor=FunctionExtractor(extractor_fn or regex_extract,
                                    "regex-naive"),
        conditions=[
            Condition.parse("elements.allergy_flag.recall >= 0.85"),
            Condition.parse(
                "distractors.discontinued_medication.fp_rate <= 0.10"),
        ],
        rules=reference_rules(),
        backend=backend,
    )


# ===================================================================
# The reference TABLE and a naive cleaner (tabular counterparts of
# reference_spec and regex_extract).
# ===================================================================

def reference_table(rows: int = 200, master_seed: int = 7):
    """Encounter billing extract: bimodal stays, cost derived from
    stay, discharge after visit, and every mess tier including
    format-valid wrong values."""
    pass  # intra-package import inlined above
    return TableSpec(
        title="Encounter billing extract",
        rows=rows,
        master_seed=master_seed,
        duplicate_rate=0.1,
        columns=[
            ColumnSpec("patient_id", "str_id",
                       {"kind": "sequence", "prefix": "PT-",
                        "start": 5000}),
            ColumnSpec("patient_name", "person_name", {},
                       ColumnMess(case_rate=0.3, space_rate=0.2)),
            ColumnSpec("age", "int",
                       {"kind": "normal", "mean": 58, "std": 18,
                        "min": 0, "max": 105},
                       ColumnMess(missing_rate=0.15)),
            ColumnSpec("department", "category",
                       {"kind": "categorical",
                        "choices": ["cardiology", "oncology",
                                    "orthopedics", "emergency"],
                        "weights": [5, 2, 2, 1]},
                       ColumnMess(typo_rate=0.2)),
            ColumnSpec("los_days", "int",
                       {"kind": "mixture",
                        "components": [
                            {"kind": "uniform", "min": 1,
                             "max": 4},
                            {"kind": "normal", "mean": 18,
                             "std": 5, "min": 8, "max": 45},
                        ],
                        "weights": [0.7, 0.3]}),
            ColumnSpec("total_cost", "float",
                       {"kind": "lognormal", "mu": 7.5,
                        "sigma": 0.8, "min": 50},
                       ColumnMess(outlier_rate=0.05,
                                  outlier_factor=100.0,
                                  wrong_rate=0.1)),
            ColumnSpec("visit_date", "date",
                       {"kind": "date_range",
                        "start": "2026-01-01",
                        "end": "2026-06-30"},
                       ColumnMess(format_rate=0.4,
                                  wrong_rate=0.08)),
            ColumnSpec("discharge_date", "date"),
            ColumnSpec("active", "bool",
                       {"kind": "bernoulli", "p": 0.7},
                       ColumnMess(format_rate=0.3)),
        ],
        rules=[
            {"kind": "date_after", "earlier": "visit_date",
             "later": "discharge_date",
             "days_from": "los_days"},
            {"kind": "derived", "target": "total_cost",
             "source": "los_days", "factor": 1150.0,
             "noise_sigma": 0.2},
        ],
    )


def strip_cleaner(rows):
    """The naive tabular cleaner: strips whitespace, nothing else.
    Fixes exactly the `space` op; misses everything; detects no
    wrong values — the honest baseline for cleaning demos."""
    return [{k: str(v).strip() for k, v in row.items()}
            for row in rows]


---
# Walkthrough

Everything below runs deterministically (StubBackend).
Flip `RUN_BEDROCK = True` for realistic prose (needs
`bedrock:InvokeModel` on the execution role).


In [ ]:
RUN_BEDROCK = False
BEDROCK_MODEL_ID = "anthropic.claude-sonnet-4-6-v1:0"
BEDROCK_REGION = "us-west-2"
CORPUS_DIR = "corpus/notebook_run_001"


## 1. Spec and plan


In [ ]:
spec = reference_spec(size=20, master_seed=42)
spec.validate()
blueprints = plan_corpus(spec)
import json as _json
print(_json.dumps(corpus_stats(blueprints), indent=2))


## 2. Verified render (deterministic)


In [ ]:
documents, render_report = render_corpus(
    spec, blueprints, StubBackend())
print(render_report.format_text())
print()
print(documents["doc_00000"][:400])


## 3. Evaluate the naive extractor

Replace `regex_extract` with an adapter around any real
model: `(doc_id, text) -> [Extraction]`.


In [ ]:
report = evaluate(
    blueprints, documents,
    FunctionExtractor(regex_extract, "regex-naive"),
    reference_rules())
print(report.format_text())


## 4. A measured experiment


In [ ]:
exp = Experiment(
    name="allergy bar + discontinued-med trap",
    spec=spec,
    extractor=FunctionExtractor(regex_extract,
                                "regex-naive"),
    conditions=[
        Condition.parse(
            "elements.allergy_flag.recall >= 0.85"),
        Condition.parse(
            "distractors.discontinued_medication"
            ".fp_rate <= 0.10"),
    ],
    rules=reference_rules(),
)
result = run_experiment(exp)
print(result.finding())


## 5. Persist and reload with integrity


In [ ]:
from pathlib import Path as _P
run_dir = write_corpus(_P(CORPUS_DIR), spec,
                       blueprints, documents,
                       render_report, "stub")
_s2, _b2, _d2, _m = load_corpus(run_dir)
assert _d2 == documents
print("corpus verified ->", run_dir)


## 6. In-notebook open-source LLM (gated)

Generation INSIDE this notebook's process — no server,
no service. First run downloads weights from the
Hugging Face hub (~1-3GB; use a local/S3 path as
`model_id` in air-gapped environments). The 0.5B model
runs on CPU instances; prefer a GPU instance for 1.5B+.
The verifier + retry + fallback wrap it like any
backend — a weak model degrades measurably, never
breaks the corpus.


In [ ]:
RUN_LOCAL_LLM = False
LOCAL_MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
if RUN_LOCAL_LLM:
    import sys as _sys
    !{_sys.executable} -m pip install -q transformers torch accelerate
    hf_backend = HFLocalBackend(model_id=LOCAL_MODEL_ID)
    hf_docs, hf_rr = render_corpus(spec, blueprints,
                                   hf_backend)
    print(hf_rr.format_text())
    print(evaluate(
        blueprints, hf_docs,
        FunctionExtractor(regex_extract,
                          "regex-naive"),
        reference_rules()).format_text())
else:
    print("RUN_LOCAL_LLM is False — skipped.")


## 7. Bedrock (gated) — realistic prose


In [ ]:
if RUN_BEDROCK:
    backend = BedrockBackend(
        model_id=BEDROCK_MODEL_ID,
        region=BEDROCK_REGION)
    live_docs, live_rr = render_corpus(
        spec, blueprints, backend)
    print(live_rr.format_text())
    print(evaluate(
        blueprints, live_docs,
        FunctionExtractor(regex_extract,
                          "regex-naive"),
        reference_rules()).format_text())
else:
    print("RUN_BEDROCK is False — skipped.")


## 8. Self-validation

A compact assertion suite over the inlined library —
the notebook proves itself on every full run.


In [ ]:
corpus_b = plan_corpus(reference_spec(size=20,
                                      master_seed=42))
assert all(a.to_json() == b.to_json()
           for a, b in zip(blueprints, corpus_b)), \
    "determinism"
def _perfect(doc_id, text):
    bp = next(b for b in blueprints
              if b.doc_id == doc_id)
    cats = {"current_medication":
            "current medication",
            "followup_appointment": "follow-up",
            "allergy_flag": "allergy"}
    return [Extraction(cats[e.element_id],
                       e.value or e.phrasing)
            for e in bp.notes[0].elements]
_r = evaluate(blueprints, documents,
              FunctionExtractor(_perfect, "perfect"),
              reference_rules())
assert abs(_r.overall_recall - 1.0) < 1e-9, \
    "perfect recall"
assert _r.distractors[
    "discontinued_medication"].false_positives == 0
assert render_report.fallbacks == 0, "verified render"
v = verify_note(documents["doc_00000"],
                blueprints[0].notes[0],
                spec.unstructured_fields[0])
assert v.ok, "verifier"
tbp = plan_table(reference_table(rows=40))
tbp2 = plan_table(reference_table(rows=40))
assert tbp.dirty_rows == tbp2.dirty_rows, "table determinism"
trep = evaluate_cleaning(
    tbp, strip_cleaner(tbp.dirty_rows), "strip")
assert trep.ops["space"].fix_rate == 1.0
assert trep.overcorrection_rate == 0.0
camp = compile_campaign("clean", reference_table(rows=30))
assert len(camp.tiers) == 3
t1 = TableSpec.from_json(camp.tiers[0].spec_json)
assert all(c.mess.wrong_rate == 0.0 for c in t1.columns)
assert auroc([0.1, 0.9], [0, 1]) == 1.0
enc = FeatureEncoder().fit([{"a": "$1,200"}, {"a": "90"}])
assert enc.columns[0].kind == "numeric"
cleaned = autoclean(tbp.dirty_rows)
assert len(cleaned) == len(tbp.dirty_rows)
print("SELF-VALIDATION: all assertions passed.")
